# บทที่ 4 — Descriptive Statistics: สถานภาพการวิจัยภายใต้แผนงาน อพท.

Notebook นี้จัดลำดับการวิเคราะห์ตามประเด็นในเอกสาร Word ตั้งแต่ **4.1–4.7** และใช้ข้อมูลจาก Excel เฉพาะชีต **`สถานภาพแผนงาน -Clean`**

**หลักการแสดงผล**
- จำนวนแสดงเป็น `n (%)`
- Multiple response ใช้จำนวนโครงการทั้งหมดเป็นตัวหาร จึงรวมร้อยละอาจเกิน 100%
- โครงการที่ใช้วิเคราะห์ถูกคัดจากแถวที่มีทั้ง **งบประมาณ** และ **สถานะงาน** เพื่อกันแถวบันทึกท้ายชีตออก
- ไม่แก้ไขไฟล์ Excel ต้นฉบับ
- กราฟบันทึกอัตโนมัติในโฟลเดอร์ `chapter4_outputs/`


**Version 2:** เพิ่ม Publication-ready chart configuration สำหรับแก้ display label, wrap ข้อความ, สี, ขนาดภาพ และลำดับหมวด โดยไม่เปลี่ยนค่าจริงใน Excel

In [ ]:
# 0) Imports & configuration
from pathlib import Path
import re
import math
import warnings
import textwrap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
from IPython.display import display, Markdown

warnings.filterwarnings("ignore", category=UserWarning)

SHEET_NAME = "สถานภาพแผนงาน -Clean"
OUTPUT_DIR = Path("chapter4_outputs_v3")
OUTPUT_DIR.mkdir(exist_ok=True)

# รองรับทั้งกรณีเปิด notebook ในโฟลเดอร์เดียวกับ Excel และใน /mnt/data
candidates = [
    Path("ตารางสถานภาพงานวิจัย-final05Sep2026.xlsx"),
    Path("/mnt/data/ตารางสถานภาพงานวิจัย-final05Sep2026.xlsx"),
]
DATA_PATH = next((p for p in candidates if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "ไม่พบไฟล์ Excel กรุณาวาง 'ตารางสถานภาพงานวิจัย-final05Sep2026.xlsx' "
        "ไว้ในโฟลเดอร์เดียวกับ notebook"
    )

# เลือก font ไทยที่มีในเครื่องโดยอัตโนมัติ
preferred_fonts = ["Noto Sans Thai", "Tahoma", "Leelawadee UI", "Arial Unicode MS", "DejaVu Sans"]
available_fonts = {f.name for f in font_manager.fontManager.ttflist}
thai_font = next((f for f in preferred_fonts if f in available_fonts), "DejaVu Sans")

plt.rcParams.update({
    "font.family": [thai_font, "DejaVu Sans"],
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.labelsize": 11,
    "figure.dpi": 130,
    "savefig.dpi": 220,
    "axes.unicode_minus": False,
})

print("DATA:", DATA_PATH)
print("SHEET:", SHEET_NAME)
print("FONT:", thai_font)




## 1) อ่านข้อมูลและเตรียมชุดวิเคราะห์

ไฟล์นี้มีหัวตาราง 2 ชั้น จึงอ่านด้วย `header=[0,1]` แล้วใช้ชื่อหัวข้อชั้นล่างเป็นชื่อคอลัมน์หลัก  
การคัด 27 โครงการจะอาศัยแถวที่มี **งบประมาณที่ได้รับจัดสรร** และ **สถานะงาน** ไม่เป็นค่าว่าง


In [ ]:
raw = pd.read_excel(DATA_PATH, sheet_name=SHEET_NAME, header=[0, 1])

def flatten_columns(columns):
    names = []
    used = {}
    for top, bottom in columns:
        bottom = str(bottom)
        top = str(top)
        base = top.strip() if bottom.startswith("Unnamed:") else bottom.strip()
        base = re.sub(r"\s+", " ", base)
        count = used.get(base, 0)
        used[base] = count + 1
        names.append(base if count == 0 else f"{base}__{count+1}")
    return names

raw.columns = flatten_columns(raw.columns)

# แปลงงบประมาณเป็นตัวเลขเพื่อใช้เป็นเงื่อนไขคัดแถวโครงการจริง
raw["งบประมาณที่ได้รับจัดสรร"] = pd.to_numeric(
    raw["งบประมาณที่ได้รับจัดสรร"], errors="coerce"
)

df = raw[
    raw["งบประมาณที่ได้รับจัดสรร"].notna()
    & raw["สถานะงาน"].notna()
].copy()

df = df.reset_index(drop=True)

print(f"จำนวนแถวในชีตทั้งหมด: {len(raw):,}")
print(f"จำนวนโครงการที่ใช้วิเคราะห์: {len(df):,}")
display(df[["รหัสโครงการ", "ชื่อโครงการภาษาไทย", "งบประมาณที่ได้รับจัดสรร", "สถานะงาน"]].head())


## 2) Chart configuration — ปรับชื่อ สี และสัดส่วนรูปจากจุดเดียว

ส่วนนี้ออกแบบให้ **ค่าจริงสำหรับการวิเคราะห์ไม่ถูกแก้** แต่สามารถเปลี่ยนชื่อที่แสดงบนกราฟ (`display label`) ได้อิสระ

วิธีใช้หลัก ๆ:
- แก้คำยาวใน `*_LABELS` ให้เป็นคำสั้นที่ต้องการแสดง
- ปรับสีใน `COLORS`
- ปรับขนาดตัวอักษร/ความละเอียดใน `CHART_STYLE`
- ถ้าชื่อยังยาว ให้ปรับ `wrap_width` ตอนเรียก `barh_count()`

> ตารางและการคำนวณยังใช้ค่าต้นฉบับจาก Excel เสมอ การย่อชื่อมีผลเฉพาะรูปเท่านั้น


In [ ]:
# ============================================================
# PUBLICATION-READY CHART CONFIGURATION
# แก้ตรงนี้ก่อนสร้างรูปได้เลย โดยไม่กระทบค่าคำนวณจริง
# ============================================================

COLORS = {
    "size": "#8FB9E0",         # ฟ้าอ่อน
    "status": "#A8D5BA",       # เขียวอ่อน
    "duration": "#F6C28B",     # ส้มพีชอ่อน
    "affiliation": "#C7B6E5",  # ม่วงอ่อน
    "team": "#F2B5B5",         # ชมพูอ่อน
    "pmu": "#9ED9CC",          # เขียวมิ้นต์
    "research_type": "#F7D6A3",# ครีมส้มอ่อน
    "oecd": "#AFCBFF",         # ฟ้านม
    "output": "#F4B6C2",       # ชมพูพาสเทล
    "trl": "#C9B6E4",          # ม่วงลาเวนเดอร์
    "srl": "#D9C2F0",          # ม่วงอ่อนมาก
    "academic": "#A7C7E7",     # ฟ้าเทาอ่อน
    "actual_users": "#B7D88C", # เขียวใบไม้อ่อน
    "target_users": "#A2D2A2", # เขียวเซจอ่อน
    "outcome": "#F7C59F",      # พีชอ่อน
    "impact": "#F5A97F",       # ส้มอ่อน
    "primary": "#8FB9E0",      # alias เดิม
    "secondary": "#A2D2A2",    # alias เดิม
    "accent": "#F5A97F",       # alias เดิม
    "purple": "#C9B6E4",       # alias เดิม
    "neutral": "#D9D9D9",      # เทาอ่อน
    "dark": "#4A4A4A",
}

CHART_COLOR_MAP = {
    "4_2_1_project_size_count.png": COLORS["size"],
    "4_2_2_close_status.png": COLORS["status"],
    "4_2_2_duration_groups.png": COLORS["duration"],
    "4_2_3_pi_affiliations.png": COLORS["affiliation"],
    "4_2_3_researcher_team_size.png": COLORS["team"],
    "4_3_2_pmu_research_framework.png": COLORS["pmu"],
    "4_3_2_research_type.png": COLORS["research_type"],
    "4_3_3_oecd_research_field.png": COLORS["oecd"],
    "4_4_1_output_types.png": COLORS["output"],
    "4_4_2_trl.png": COLORS["trl"],
    "4_4_3_srl.png": COLORS["srl"],
    "4_5_1_academic_benefits.png": COLORS["academic"],
    "4_6_1_1_actual_users.png": COLORS["actual_users"],
    "4_6_1_1_target_users.png": COLORS["target_users"],
    "4_6_1_2_outcome_types.png": COLORS["outcome"],
    "4_6_2_impact_dimensions.png": COLORS["impact"],
}


CHART_STYLE = {
    "title_size": 16,
    "axis_label_size": 12,
    "tick_size": 11,
    "annotation_size": 11,
    "grid_alpha": 0.14,
    "save_dpi": 240,
    "default_width": 10,
    "default_height": 5.5,
    "bar_height_per_item": 0.55,
}

# ------------------------------------------------------------------
# Display labels: ซ้าย = ค่าจริงใน Excel / ขวา = คำสั้นสำหรับรูป
# ถ้าหา key ไม่เจอ ระบบจะใช้ข้อความเดิมและ wrap ให้อัตโนมัติ
# ------------------------------------------------------------------

PMU_LABELS = {
    "การพัฒนากลไกการจัดบริการสาธารณะขององค์กรปกครองส่วนท้องถิ่น (User คือ ช่วยชาวบ้านในพื้นที่)": "พัฒนากลไกบริการสาธารณะของ อปท.",
    "การพัฒนาเทคโนโลยีดิจิทัล (User คือ อปท. เพื่อพัฒนาการทำงาน)": "พัฒนาเทคโนโลยีดิจิทัลของ อปท.",
    "การพัฒนากลไกและกระบวนการสร้างการเปลี่ยนแปลงเพื่อเพิ่มรายได้ของท้องถิ่น": "กลไกเพิ่มรายได้ของท้องถิ่น",
    "การพัฒนาศักยภาพเชิงสถาบันและกรอบกฎหมาย": "ศักยภาพเชิงสถาบันและกฎหมาย",
    "เป็นการพัฒนาสมรรถนะ?การบริหารจัดการน้ำ ซึ่งเป็นบริการสาธารณะของ อปท.": "สมรรถนะการบริหารจัดการน้ำ",
    "เป็นการพัฒนาศักยภาพนักวิจัย เพื่อตอบสนองต่อการพัฒนาปสก.การทำงานของ อปท.": "พัฒนาศักยภาพนักวิจัยเพื่อ อปท.",
    "การพัฒนาเทคโนโลยีดิจิทัล": "พัฒนาเทคโนโลยีดิจิทัล",
}

OUTPUT_LABELS = {
    "ฐานข้อมูล ระบบและกลไก": "ฐานข้อมูล ระบบและกลไก",
    "กำลังคน หรือหน่วยงาน ที่ได้รับการพัฒนาทักษะ": "กำลังคน/หน่วยงานที่ได้รับการพัฒนาทักษะ",
    "ข้อเสนอแนะเชิงนโยบาย:ข้อเสนอที่มุ่งใช้ประกอบการตัดสินใจ การกำหนดนโยบาย มาตรการ แผนงาน แนวทางปฏิบัติ หรือกฎเกณฑ์ของหน่วยงานหรือองค์กร": "ข้อเสนอแนะเชิงนโยบาย",
    "เครื่องมือ และโครงสร้างพื้นฐานที่สร้างขึ้น หรือพัฒนาต่อยอดภายใต้โครงการ": "เครื่องมือ/โครงสร้างพื้นฐาน",
}

USER_LABELS = {
    "หน่วยงานภาครัฐในระดับพื้นที่ ตำบล อำเภอ จังหวัด": "หน่วยงานภาครัฐระดับพื้นที่",
    "หน่วยงานรัฐระดับส่วนกลาง/ผู้กำหนดนโยบาย": "หน่วยงานส่วนกลาง/ผู้กำหนดนโยบาย",
    "บุคลากรทางการศึกษา/สถาบันการศึกษา": "บุคลากร/สถาบันการศึกษา",
}

OUTCOME_LABELS = {
    "ลดความสูญเสียทางเศรษฐกิจสังคม เช่น ลดอัตราการเจ็บป่วย ลดอัตราการตาย และลดค่าใช้จ่ายด้านสุขภาพ เป็นต้น": "ลดความสูญเสียทางเศรษฐกิจและสังคม",
    "เพิ่มผลิตภาพการผลิต/เพิ่มผลผลิต/เพิ่มประสิทธิภาพการดำเนินงาน": "เพิ่มผลิตภาพ/ประสิทธิภาพการดำเนินงาน",
    "พัฒนาคุณภาพชีวิต (เพราะเข้าถึงบริการสุขภาพคุณภาพมาตรฐาน)": "พัฒนาคุณภาพชีวิต",
    "คุณภาพสิ่งแวดล้อมดีขึ้น ลดมลพิษ/มลภาวะ/ลดก๊าซเรือนกระจก": "คุณภาพสิ่งแวดล้อมดีขึ้น",
}

ACADEMIC_LABELS = {
    "จำนวนครั้งการเผยแพร่ผ่านการอบรม/สัมมนา/เวทีสาธารณะ/นิทรรศการ": "อบรม/สัมมนา/เวทีสาธารณะ/นิทรรศการ",
    "จำนวนสื่อ clip vdo หรือเพจเผยแพร่/งานเขียนออนไลน์": "สื่อ/คลิปวิดีโอ/เพจ/งานเขียนออนไลน์",
    "จำนวนครั้งการเผยแพร่ผ่าน วิดีทัศน์ โทรทัศน์ วิทยุ นสพ. อินเตอร์เน็ต": "เผยแพร่ผ่านสื่อมวลชน/อินเทอร์เน็ต",
    "มีการใช้ประโยชน์กับการเรียนการสอน (จำนวนวิชา)": "ใช้ประโยชน์ในการเรียนการสอน",
    "มีการใช้ประโยชน์กับการวิจัยเพื่อพัฒนานิสิต (จำนวนนิสิต ที่ทำวิจัยหรือวิทยานิพนธ์)": "ใช้พัฒนางานวิจัย/วิทยานิพนธ์นิสิต",
    "จำนวนบทความ (นำเสนอในที่ประชุมระดับนานาชาติ)": "บทความนำเสนอประชุมระดับนานาชาติ",
    "จำนวนบทความ (นำเสนอในที่ประชุมระดับประเทศ)": "บทความนำเสนอประชุมระดับประเทศ",
}

OECD_FIELD_LABELS = {
    "5. สังคมศาสตร์ (Social Sciences)": "สังคมศาสตร์",
    "2. วิศวกรรมและเทคโนโลยี (Engineering and technology)": "วิศวกรรมและเทคโนโลยี",
    "3. วิทยาศาสตร์การแพทย์และสุขภาพ (Medical and Health Sciences)": "วิทยาศาสตร์การแพทย์และสุขภาพ",
}

# กรณีอยากแก้ชื่อสังกัดเฉพาะแห่ง ให้เติมใน dict นี้
AFFILIATION_LABELS = {
    "โรงพยาบาลวชิระ(วชิรพยาบาล) (ย้ายไปใต้คณะแพทยศาสตร์วชิรพยาบาล มหาวิทยาลัยนวมินทราธิราช)": "วชิรพยาบาล ม.นวมินทราธิราช",
}


def shorten_display_label(value, label_map=None, wrap_width=30):
    """คืนชื่อสำหรับกราฟเท่านั้น: map ก่อน แล้วค่อยตัดบรรทัดให้พอดี"""
    text = str(value)
    if label_map:
        text = label_map.get(text, text)
    if wrap_width:
        text = "\n".join(textwrap.wrap(text, width=wrap_width, break_long_words=False, break_on_hyphens=False))
    return text


def apply_display_labels(index, label_map=None, wrap_width=30):
    return [shorten_display_label(x, label_map=label_map, wrap_width=wrap_width) for x in index]




In [ ]:
# Helper functions สำหรับตารางและกราฟ
def clean_text(x):
    if pd.isna(x):
        return np.nan
    x = str(x).replace("\n", " ").replace("\r", " ")
    x = re.sub(r"\s+", " ", x).strip()
    return x if x else np.nan
def n_pct(n, denom):
    if denom == 0:
        return f"{int(n):,} (0.0%)"
    return f"{int(n):,} ({n/denom*100:.1f}%)"
def frequency_table(series, denom=None, dropna=True, sort=True):
    s = series.map(clean_text)
    counts = s.value_counts(dropna=dropna)
    if denom is None:
        denom = s.notna().sum()
    out = pd.DataFrame({"จำนวน": counts})
    out["ร้อยละ"] = out["จำนวน"] / denom * 100
    out["n (%)"] = [n_pct(n, denom) for n in out["จำนวน"]]
    if sort:
        out = out.sort_values(["จำนวน"], ascending=False)
    return out
def barh_count(
    table,
    title,
    xlabel="จำนวนโครงการ",
    filename=None,
    pct_col="ร้อยละ",
    *,
    label_map=None,
    wrap_width=24,
    color=None,
    figsize=None,
    order=None,
    sort_by_count=True,
    ylim_pad=1.18,
    note=None,
):
    """
    Vertical bar chart
    - แกน X = หมวดหมู่
    - แกน Y = จำนวน
    - แต่ละแท่งใช้สี pastel แตกต่างกัน
    """

    plot_df = table.copy()

    # ---------------------------------------------------------
    # จัดลำดับข้อมูล
    # ---------------------------------------------------------
    if order is not None:
        existing = [x for x in order if x in plot_df.index]
        remainder = [x for x in plot_df.index if x not in existing]
        plot_df = plot_df.reindex(existing + remainder)

    elif sort_by_count:
        plot_df = plot_df.sort_values(
            "จำนวน",
            ascending=False
        )

    # ---------------------------------------------------------
    # ชื่อที่ใช้แสดงบนกราฟ
    # ไม่แก้ข้อมูลต้นฉบับ
    # ---------------------------------------------------------
    display_index = apply_display_labels(
        plot_df.index,
        label_map=label_map,
        wrap_width=wrap_width
    )

    # ---------------------------------------------------------
    # ขนาดภาพ
    # ---------------------------------------------------------
    n_cat = len(plot_df)

    if figsize is None:
        width = max(
            9,
            min(18, n_cat * 1.2 + 5)
        )

        height = 7

        figsize = (width, height)

    # ---------------------------------------------------------
    # สี Pastel
    # ---------------------------------------------------------
    PASTEL_BAR_COLORS = [
        "#8FB9E0",   # ฟ้าอ่อน
        "#A8D5BA",   # เขียวอ่อน
        "#F6C28B",   # พีช
        "#C7B6E5",   # ม่วงอ่อน
        "#F2B5B5",   # ชมพู
        "#9ED9CC",   # มิ้นต์
        "#F7D6A3",   # ครีมส้ม
        "#AFCBFF",   # ฟ้านม
        "#F4B6C2",   # ชมพูอ่อน
        "#B7D88C",   # เขียวอ่อน
        "#D9C2F0",   # ลาเวนเดอร์
        "#FFD6A5",   # apricot
    ]

    # ให้แต่ละแท่งใช้สีต่างกัน
    bar_colors = [
        PASTEL_BAR_COLORS[
            i % len(PASTEL_BAR_COLORS)
        ]
        for i in range(n_cat)
    ]

    # ---------------------------------------------------------
    # สร้างกราฟ
    # ---------------------------------------------------------
    fig, ax = plt.subplots(
        figsize=figsize
    )

    bars = ax.bar(
        range(n_cat),
        plot_df["จำนวน"].values,
        color=bar_colors,
        edgecolor="white",
        linewidth=1
    )

    # ---------------------------------------------------------
    # Title
    # ---------------------------------------------------------
    ax.set_title(
        title,
        fontsize=CHART_STYLE.get(
            "title_size",
            18
        ),
        pad=15,
        fontweight="bold"
    )

    # ---------------------------------------------------------
    # Axis
    # ---------------------------------------------------------
    ax.set_ylabel(
        xlabel,
        fontsize=CHART_STYLE.get(
            "axis_label_size",
            13
        )
    )

    ax.set_xlabel("")

    ax.set_xticks(
        range(n_cat)
    )

    ax.set_xticklabels(
        display_index,
        fontsize=CHART_STYLE.get(
            "tick_size",
            11
        )
    )

    # ---------------------------------------------------------
    # หมุน label ถ้าข้อความยาว
    # ---------------------------------------------------------
    max_label_len = max(
        [
            len(
                str(x).replace(
                    "\n",
                    ""
                )
            )
            for x in display_index
        ]
    ) if n_cat > 0 else 0

    if (
        n_cat >= 6
        or max_label_len > 15
    ):

        plt.setp(
            ax.get_xticklabels(),
            rotation=25,
            ha="right"
        )

    else:

        plt.setp(
            ax.get_xticklabels(),
            rotation=0,
            ha="center"
        )

    # ---------------------------------------------------------
    # Grid
    # ---------------------------------------------------------
    ax.grid(
        axis="y",
        linestyle="--",
        alpha=0.20
    )

    ax.set_axisbelow(True)

    ax.spines[
        ["top", "right"]
    ].set_visible(False)

    # ---------------------------------------------------------
    # Scale Y
    # ---------------------------------------------------------
    max_val = max(
        float(
            plot_df["จำนวน"].max()
        ),
        1
    )

    ax.set_ylim(
        0,
        max_val * ylim_pad + 0.5
    )

    # ---------------------------------------------------------
    # จำนวน + %
    # บนยอดแท่ง
    # ---------------------------------------------------------
    for bar, (_, row) in zip(
        bars,
        plot_df.iterrows()
    ):

        n = int(
            row["จำนวน"]
        )

        if pct_col in row.index:

            label = (
                f"{n} "
                f"({row[pct_col]:.1f}%)"
            )

        else:

            label = str(n)

        ax.text(
            bar.get_x()
            + bar.get_width() / 2,

            bar.get_height()
            + max_val * 0.02,

            label,

            ha="center",
            va="bottom",

            fontsize=CHART_STYLE.get(
                "annotation_size",
                11
            )
        )

    # ---------------------------------------------------------
    # Note ด้านล่าง
    # ---------------------------------------------------------
    if note:

        fig.text(
            0.01,
            0.01,
            note,
            ha="left",
            va="bottom",
            fontsize=9,
            color="#555555"
        )

        fig.tight_layout(
            rect=[
                0,
                0.05,
                1,
                1
            ]
        )

    else:

        fig.tight_layout()

    # ---------------------------------------------------------
    # Save
    # ---------------------------------------------------------
    if filename:

        fig.savefig(
            OUTPUT_DIR / filename,
            dpi=CHART_STYLE.get(
                "save_dpi",
                240
            ),
            bbox_inches="tight"
        )

    plt.show()

    return fig
def save_table(table, filename):
    table.to_csv(OUTPUT_DIR / filename, encoding="utf-8-sig")
def numeric_prefix(series):
    """ดึงเลขนำหน้าจากข้อความ เช่น '6. ต้นแบบ...' -> 6"""
    return pd.to_numeric(
        series.astype(str).str.extract(r"^\s*([0-9]+)", expand=False),
        errors="coerce"
    )
N = len(df)
print("N =", N)



# 4.1 ข้อมูลเบื้องต้นของแผนงานวิจัย

ส่วน 4.1 ใน Word เป็นบริบทเชิงนโยบาย ได้แก่ นิยามแผน/แพลตฟอร์ม/โปรแกรม, OKR และโครงสร้างแผนงาน  
Notebook จึงใช้ส่วนนี้เป็น **บริบทประกอบ** และเริ่ม descriptive statistics เชิงปริมาณเต็มรูปแบบตั้งแต่ 4.2

> ตัวเลขภาพรวมด้านจำนวนโครงการและงบประมาณตรวจจากฐานข้อมูลอีกครั้งด้านล่าง


In [ ]:
# 4.1 ภาพรวมเชิงตัวเลข
overview = pd.DataFrame({
    "ตัวชี้วัด": [
        "จำนวนโครงการทั้งหมด",
        "งบประมาณรวม (บาท)",
        "งบประมาณเฉลี่ยต่อโครงการ (บาท)",
        "ค่ามัธยฐานงบประมาณต่อโครงการ (บาท)"
    ],
    "ค่า": [
        N,
        df["งบประมาณที่ได้รับจัดสรร"].sum(),
        df["งบประมาณที่ได้รับจัดสรร"].mean(),
        df["งบประมาณที่ได้รับจัดสรร"].median()
    ]
})
display(overview)


# 4.2 ปัจจัยนำเข้า

ตามกรอบ Word ส่วนนี้ **คิดรวมโครงการที่ขยายเวลาและยุติ** เพื่อสะท้อนทรัพยากรทั้งหมดที่ใช้ขับเคลื่อนแผนงาน


## 4.2.1 งบประมาณ

กราฟแนะนำ: **horizontal bar chart** เพราะอ่านความแตกต่างระหว่างขนาดโครงการได้ง่าย และสามารถใส่ทั้ง `จำนวน (%)` กับงบเฉลี่ยได้โดยไม่แน่นเกินไป


In [ ]:
budget = df["งบประมาณที่ได้รับจัดสรร"]

budget_desc = pd.Series({
    "งบประมาณรวม (ล้านบาท)": budget.sum()/1e6,
    "งบประมาณเฉลี่ย (ล้านบาท/โครงการ)": budget.mean()/1e6,
    "มัธยฐาน (ล้านบาท/โครงการ)": budget.median()/1e6,
    "ต่ำสุด (ล้านบาท)": budget.min()/1e6,
    "สูงสุด (ล้านบาท)": budget.max()/1e6,
})
display(budget_desc.to_frame("ค่า").round(3))

size_col = "ขนาดโครงการ"
size_table = (
    df.groupby(size_col, dropna=False)
      .agg(
          n_projects=("รหัสโครงการ", "count"),
          budget_total=("งบประมาณที่ได้รับจัดสรร", "sum"),
          budget_mean=("งบประมาณที่ได้รับจัดสรร", "mean"),
      )
      .rename(columns={"n_projects":"จำนวน", "budget_total":"งบประมาณรวม", "budget_mean":"งบประมาณเฉลี่ย"})
)
size_table["ร้อยละ"] = size_table["จำนวน"] / N * 100
size_table["สัดส่วนงบประมาณ"] = size_table["งบประมาณรวม"] / budget.sum() * 100
size_table["n (%)"] = [n_pct(n, N) for n in size_table["จำนวน"]]
size_table["งบรวม (ล้านบาท)"] = size_table["งบประมาณรวม"]/1e6
size_table["งบเฉลี่ย (ล้านบาท)"] = size_table["งบประมาณเฉลี่ย"]/1e6

display(size_table[["n (%)", "งบรวม (ล้านบาท)", "สัดส่วนงบประมาณ", "งบเฉลี่ย (ล้านบาท)"]].round(2))
save_table(size_table, "4_2_1_budget_by_project_size.csv")

barh_count(
    size_table[["จำนวน", "ร้อยละ"]],
    "จำนวนโครงการจำแนกตามขนาดโครงการ",
    filename="4_2_1_project_size_count.png",
    color=COLORS["primary"],
    wrap_width=24
)


## 4.2.2 ระยะเวลาและสถานะโครงการ

แสดงทั้ง
1. การกระจายตามระยะเวลา: ต่ำกว่า 1 ปี / 1 ปี / มากกว่า 1 ปี  
2. จำนวนโครงการขยายเวลาและยุติ  
3. สถานะปิดโครงการ โดยคำนวณร้อยละจาก **โครงการที่ไม่ยุติ** ตามกรอบ Word


In [ ]:
# ระยะเวลาใช้ปีและเดือนตามจริง
years = pd.to_numeric(df["ระยะเวลาปี"], errors="coerce").fillna(0)
months = pd.to_numeric(df["ระยะเวลาเดือน"], errors="coerce").fillna(0)
duration_months = years*12 + months

df["ระยะเวลารวม_เดือน"] = duration_months

def duration_group(m):
    if pd.isna(m):
        return "ไม่ระบุ"
    if m < 12:
        return "ต่ำกว่า 1 ปี"
    if m == 12:
        return "1 ปี"
    return "มากกว่า 1 ปี"

df["กลุ่มระยะเวลา"] = df["ระยะเวลารวม_เดือน"].map(duration_group)

duration_order = ["ต่ำกว่า 1 ปี", "1 ปี", "มากกว่า 1 ปี", "ไม่ระบุ"]
duration_table = frequency_table(df["กลุ่มระยะเวลา"], denom=N).reindex(
    [x for x in duration_order if x in set(df["กลุ่มระยะเวลา"])]
)

display(pd.Series({
    "ระยะเวลาเฉลี่ย (เดือน)": df["ระยะเวลารวม_เดือน"].mean(),
    "มัธยฐาน (เดือน)": df["ระยะเวลารวม_เดือน"].median(),
    "ต่ำสุด (เดือน)": df["ระยะเวลารวม_เดือน"].min(),
    "สูงสุด (เดือน)": df["ระยะเวลารวม_เดือน"].max(),
}).to_frame("ค่า").round(2))

display(duration_table[["จำนวน", "ร้อยละ", "n (%)"]])
save_table(duration_table, "4_2_2_duration_groups.csv")

barh_count(
    duration_table[["จำนวน", "ร้อยละ"]],
    "จำนวนโครงการจำแนกตามระยะเวลาดำเนินงาน",
    filename="4_2_2_duration_groups.png",
    color=COLORS["primary"],
    wrap_width=24
)

# ขยายเวลา
extension_months = pd.to_numeric(df["ระยะเวลาที่ขยายเวลา (month) (Gift)"], errors="coerce").fillna(0)
extended = extension_months.gt(0)
terminated = df["สถานะงาน"].astype(str).str.contains("ยุติ", na=False)

print("ขยายเวลา:", n_pct(extended.sum(), N))
print("ยุติโครงการ:", n_pct(terminated.sum(), N))

# ปิดโครงการคำนวณเฉพาะโครงการที่ไม่ยุติ
non_terminated = df.loc[~terminated].copy()
closed = non_terminated["สถานะงาน"].astype(str).str.strip().eq("ปิดโครงการ")
close_table = pd.DataFrame({
    "จำนวน": [closed.sum(), (~closed).sum()]
}, index=["ปิดโครงการ", "ยังไม่ปิดโครงการ"])
close_table["ร้อยละ"] = close_table["จำนวน"] / len(non_terminated) * 100
close_table["n (%)"] = [n_pct(n, len(non_terminated)) for n in close_table["จำนวน"]]

display(close_table)
save_table(close_table, "4_2_2_close_status_nonterminated.csv")

# ============================================================
# 4.2.2 Pie chart 2 วง
# วงที่ 1: สถานะโครงการจาก column AI
# วงที่ 2: รายละเอียดโครงการขยายเวลา
# ============================================================

# ------------------------------------------------------------
# วงที่ 1: ใช้ column AI = สถานะงาน
# แบ่งเป็น:
# 1) ปิดโครงการแล้ว
# 2) ขยายเวลา (อยู่ระหว่างดำเนินการ)
# 3) ยุติโครงการ
# ------------------------------------------------------------

status_series = df["สถานะงาน"].astype(str).str.strip()

closed_n = (status_series == "ปิดโครงการ").sum()
terminated_n = status_series.str.contains("ยุติ", na=False).sum()

# ถือว่าที่เหลือซึ่งไม่ใช่ปิดโครงการและไม่ใช่ยุติ = ขยายเวลา/อยู่ระหว่างดำเนินการ
extended_n = len(df) - closed_n - terminated_n

overall_labels = [
    "ปิดโครงการแล้ว",
    "ขยายเวลา",
    "ยุติโครงการ"
]

overall_sizes = [
    closed_n,
    extended_n,
    terminated_n
]

# ------------------------------------------------------------
# วงที่ 2: เจาะเฉพาะโครงการขยายเวลา
# ใช้จำนวนเดือนจากคอลัมน์ระยะเวลาที่ขยายเวลา
# แยกเป็น:
# - ขยาย 6 เดือน
# - ขยาย 3 เดือน
# ------------------------------------------------------------

extension_months = pd.to_numeric(
    df["ระยะเวลาที่ขยายเวลา (month) (Gift)"],
    errors="coerce"
).fillna(0)

ext_6 = (extension_months == 6).sum()
ext_3 = (extension_months == 3).sum()

ext_labels = []
ext_sizes = []

if ext_6 > 0:
    ext_labels.append("ขยาย 6 เดือน")
    ext_sizes.append(ext_6)

if ext_3 > 0:
    ext_labels.append("ขยาย 3 เดือน")
    ext_sizes.append(ext_3)

# -----------------------------
# pastel colors โทนเดิม
# -----------------------------
overall_colors = [
    "#8FB9E0",  # ฟ้าอ่อน
    "#A8D5BA",  # เขียวอ่อน
    "#F6C28B",  # พีชอ่อน
    "#D9D9D9",  # เทาอ่อน
]

extension_colors = [
    "#C7B6E5",  # ม่วงอ่อน
    "#F2B5B5",  # ชมพูอ่อน
    "#9ED9CC",  # มิ้นต์อ่อน (เผื่อมีมากกว่า 2)
]

# -----------------------------
# ฟังก์ชัน label บน pie
# -----------------------------
def autopct_with_n(values):
    total = sum(values)
    def _fmt(pct):
        n = int(round(pct * total / 100.0))
        return f"{pct:.1f}%\n({n})"
    return _fmt

# -----------------------------
# plot 2 วง
# -----------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# วงที่ 1: ภาพรวม
wedges1, texts1, autotexts1 = axes[0].pie(
    overall_sizes,
    labels=overall_labels,
    autopct=autopct_with_n(overall_sizes),
    startangle=90,
    colors=overall_colors[:len(overall_sizes)],
    pctdistance=0.72,
    labeldistance=1.08,
    wedgeprops=dict(edgecolor="white", linewidth=1)
)

axes[0].set_title(
    "ภาพรวมระยะเวลาดำเนินงานของโครงการ",
    fontsize=14,
    fontweight="bold",
    pad=14
)

# ทำให้เป็น donut ดูสวยขึ้น
centre_circle1 = plt.Circle((0, 0), 0.45, fc="white")
axes[0].add_artist(centre_circle1)

# วงที่ 2: เจาะการขยายเวลา
if len(ext_sizes) > 0:
    wedges2, texts2, autotexts2 = axes[1].pie(
        ext_sizes,
        labels=ext_labels,
        autopct=autopct_with_n(ext_sizes),
        startangle=90,
        colors=extension_colors[:len(ext_sizes)],
        pctdistance=0.72,
        labeldistance=1.08,
        wedgeprops=dict(edgecolor="white", linewidth=1)
    )

    axes[1].set_title(
        "การขยายเวลาโครงการ",
        fontsize=14,
        fontweight="bold",
        pad=14
    )

    centre_circle2 = plt.Circle((0, 0), 0.45, fc="white")
    axes[1].add_artist(centre_circle2)

else:
    axes[1].text(
        0.5, 0.5,
        "ไม่มีโครงการขยายเวลา",
        ha="center", va="center", fontsize=13
    )
    axes[1].set_title(
        "การขยายเวลาโครงการ",
        fontsize=14,
        fontweight="bold",
        pad=14
    )

# ปรับข้อความบน pie
for t in texts1 + autotexts1:
    t.set_fontsize(10)

if len(ext_sizes) > 0:
    for t in texts2 + autotexts2:
        t.set_fontsize(10)

fig.suptitle(
    "สถานะการดำเนินงานของโครงการวิจัย",
    fontsize=16,
    fontweight="bold",
    y=1.02
)

fig.tight_layout()

fig.savefig(
    OUTPUT_DIR / "4_2_2_duration_groups.png",
    dpi=CHART_STYLE.get("save_dpi", 240),
    bbox_inches="tight"
)

plt.show()


## 4.2.3 นักวิจัย คณะผู้วิจัย และหน่วยงานร่วมวิจัย

กราฟที่เหมาะ:
- จำนวนนักวิจัยต่อโครงการ → histogram/bar ตามช่วง
- สังกัดหัวหน้าโครงการ → horizontal bar โดยแสดงเฉพาะหน่วยงานที่มีโครงการ


In [ ]:
# ============================================================
# 4.3.2 ประเภทของการวิจัย + ประเภทตามกรอบ บพท.
# แบบ stacked bar ("ขนมชั้น")
#
# ความสูงแท่ง = จำนวนโครงการ
# ชั้นในแท่ง = ขนาดเล็ก / กลาง / ใหญ่
# annotation ในแต่ละชั้น = จำนวนโครงการ + งบเฉลี่ย (ล้านบาท)
# ============================================================

research_type_col = "ประเภทของการวิจัย"
size_col = "ขนาดโครงการ"
budget_col = "งบประมาณที่ได้รับจัดสรร"

# ------------------------------------------------------------
# normalize ชื่อขนาดโครงการจาก Column K
# ให้ตรงกับภาพ 4.2.1
# ------------------------------------------------------------
def normalize_project_size(x):
    if pd.isna(x):
        return np.nan

    x = str(x).strip()

    if "เล็ก" in x:
        return "ขนาดเล็ก"
    elif "กลาง" in x:
        return "ขนาดกลาง"
    elif "ใหญ่" in x:
        return "ขนาดใหญ่"
    else:
        return x

df["ขนาดโครงการ_มาตรฐาน"] = df[size_col].apply(normalize_project_size)
size_std_col = "ขนาดโครงการ_มาตรฐาน"

size_order = ["ขนาดเล็ก", "ขนาดกลาง", "ขนาดใหญ่"]

size_colors = {
    "ขนาดเล็ก": "#8FB9E0",   # ฟ้าอ่อน
    "ขนาดกลาง": "#A8D5BA",   # เขียวอ่อน
    "ขนาดใหญ่": "#F6C28B",   # พีชอ่อน
}

# ============================================================
# helper function: stacked bar
# ============================================================
def plot_stacked_count_with_budget(
    count_table,
    budget_table,
    total_pct_table,
    title,
    filename,
    label_map=None,
    wrap_width=24,
    note=None,
    figsize=None,
    rotate_xticks=0
):
    plot_counts = count_table.copy()
    plot_budgets = budget_table.copy()
    plot_totals = total_pct_table.copy()

    labels = apply_display_labels(
        plot_counts.index,
        label_map=label_map,
        wrap_width=wrap_width
    )

    n_cat = len(plot_counts)
    x = np.arange(n_cat)

    if figsize is None:
        width = max(10, min(18, n_cat * 1.5 + 4))
        height = 7.5 if n_cat <= 5 else 8.5
        figsize = (width, height)

    fig, ax = plt.subplots(figsize=figsize)

    bottom = np.zeros(n_cat)

    for size in size_order:
        if size not in plot_counts.columns:
            continue

        heights = plot_counts[size].fillna(0).values
        color = size_colors.get(size, "#D9D9D9")

        bars = ax.bar(
            x,
            heights,
            bottom=bottom,
            color=color,
            edgecolor="white",
            linewidth=1,
            width=0.64,
            label=size
        )

        # annotation ภายในแต่ละชั้น
        for i, bar in enumerate(bars):
            count_val = heights[i]

            if count_val <= 0:
                continue

            avg_budget = plot_budgets.loc[plot_counts.index[i], size] if size in plot_budgets.columns else np.nan

            # ถ้าชั้นเตี้ยมาก ให้ไม่ใส่เยอะเกินไป
            if count_val >= 2:
                label = f'{int(count_val)} โครงการ\n{avg_budget:.2f} ลบ.' if pd.notna(avg_budget) else f'{int(count_val)} โครงการ'
                ax.text(
                    bar.get_x() + bar.get_width()/2,
                    bottom[i] + count_val/2,
                    label,
                    ha="center",
                    va="center",
                    fontsize=9,
                    color="#3A3A3A"
                )
            else:
                label = f'{int(count_val)}\n{avg_budget:.2f}' if pd.notna(avg_budget) else f'{int(count_val)}'
                ax.text(
                    bar.get_x() + bar.get_width()/2,
                    bottom[i] + count_val/2,
                    label,
                    ha="center",
                    va="center",
                    fontsize=8,
                    color="#3A3A3A"
                )

        bottom += heights

    # รวมบนยอดแท่ง
    max_total = max(bottom.max(), 1)

    for i, idx in enumerate(plot_counts.index):
        total_n = int(plot_totals.loc[idx, "จำนวน"])
        total_pct = plot_totals.loc[idx, "ร้อยละ"]

        ax.text(
            x[i],
            bottom[i] + max_total * 0.02,
            f"{total_n} ({total_pct:.1f}%)",
            ha="center",
            va="bottom",
            fontsize=10,
            fontweight="bold"
        )

    ax.set_title(
        title,
        fontsize=17,
        fontweight="bold",
        pad=16
    )

    ax.set_ylabel(
        "จำนวนโครงการ",
        fontsize=13
    )

    ax.set_xlabel("")

    ax.set_xticks(x)
    ax.set_xticklabels(
        labels,
        fontsize=10
    )

    if rotate_xticks != 0:
        plt.setp(
            ax.get_xticklabels(),
            rotation=rotate_xticks,
            ha="right"
        )

    ax.grid(axis="y", linestyle="--", alpha=0.18)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)

    ax.set_ylim(0, max_total * 1.22)

    ax.legend(
        title="ขนาดโครงการ",
        frameon=False,
        fontsize=10,
        title_fontsize=10,
        loc="upper right"
    )

    if note:
        fig.text(
            0.01, 0.01,
            note,
            ha="left",
            va="bottom",
            fontsize=9,
            color="#555555"
        )
        fig.tight_layout(rect=[0, 0.04, 1, 1])
    else:
        fig.tight_layout()

    fig.savefig(
        OUTPUT_DIR / filename,
        dpi=CHART_STYLE.get("save_dpi", 240),
        bbox_inches="tight"
    )

    plt.show()


# ============================================================
# PART 1
# ประเภทของการวิจัย
# ============================================================

research_type = frequency_table(
    df[research_type_col],
    denom=N
)
display(research_type[["จำนวน", "ร้อยละ", "n (%)"]])

type_budget = (
    df.groupby(research_type_col)
      .agg(
          n_projects=("รหัสโครงการ", "count"),
          budget_mean=(budget_col, "mean"),
          budget_total=(budget_col, "sum")
      )
      .rename(columns={
          "n_projects": "จำนวน",
          "budget_mean": "งบประมาณเฉลี่ย",
          "budget_total": "งบประมาณรวม"
      })
)

type_budget["ร้อยละ"] = type_budget["จำนวน"] / N * 100
type_budget["งบเฉลี่ย (ล้านบาท)"] = type_budget["งบประมาณเฉลี่ย"] / 1e6

display(
    type_budget[["จำนวน", "ร้อยละ", "งบเฉลี่ย (ล้านบาท)"]].round(2)
)

# cross-tab นับจำนวนโครงการ แยกตามขนาด
research_count_size = pd.crosstab(
    df[research_type_col],
    df[size_std_col]
).reindex(columns=size_order, fill_value=0)

# งบเฉลี่ย แยกตามขนาด
research_budget_size = (
    df.groupby([research_type_col, size_std_col])[budget_col]
      .mean()
      .unstack()
      .reindex(columns=size_order)
      / 1e6
)

# ให้เรียงตามลำดับใน research_type
research_count_size = research_count_size.reindex(research_type.index)
research_budget_size = research_budget_size.reindex(research_type.index)

# ตารางไขว้
cross_type_size = pd.crosstab(
    df[research_type_col].map(clean_text),
    df[size_std_col].map(clean_text),
    margins=True
)
display(cross_type_size)

# plot stacked bar
plot_stacked_count_with_budget(
    count_table=research_count_size,
    budget_table=research_budget_size,
    total_pct_table=research_type[["จำนวน", "ร้อยละ"]],
    title="ประเภทของการวิจัย",
    filename="4_3_2_research_type.png",
    wrap_width=24,
    figsize=(11, 7)
)


# ============================================================
# PART 2
# ประเภทของการวิจัยตามกรอบ บพท. (Multiple response)
# ============================================================

pmu_cols = [
    "ประเภทของการวิจัยตามกรอบ บพท. (1) (Gift)",
    "ประเภทของการวิจัยตามกรอบ (2) บพท. (Gift)",
    "ประเภทของการวิจัยตามกรอบ (3) (Gift)"
]

pmu_long = (
    df[["รหัสโครงการ", size_std_col, budget_col] + pmu_cols]
      .melt(
          id_vars=["รหัสโครงการ", size_std_col, budget_col],
          value_vars=pmu_cols,
          value_name="ประเภทตามกรอบ_บพท."
      )
      .dropna(subset=["ประเภทตามกรอบ_บพท."])
)

pmu_long["ประเภทตามกรอบ_บพท."] = pmu_long["ประเภทตามกรอบ_บพท."].map(clean_text)
pmu_long = pmu_long.drop_duplicates(["รหัสโครงการ", "ประเภทตามกรอบ_บพท."])

pmu_count = pmu_long["ประเภทตามกรอบ_บพท."].value_counts()
pmu_table = pd.DataFrame({"จำนวน": pmu_count})
pmu_table["ร้อยละ"] = pmu_table["จำนวน"] / N * 100
pmu_table["n (%)"] = [n_pct(n, N) for n in pmu_table["จำนวน"]]

display(pmu_table)

pmu_budget = (
    pmu_long.groupby("ประเภทตามกรอบ_บพท.")
      .agg(
          n_projects=("รหัสโครงการ", "nunique"),
          budget_mean=(budget_col, "mean")
      )
      .rename(columns={
          "n_projects": "จำนวน",
          "budget_mean": "งบประมาณเฉลี่ย"
      })
)

pmu_budget["ร้อยละโครงการ"] = pmu_budget["จำนวน"] / N * 100
pmu_budget["งบเฉลี่ย (ล้านบาท)"] = pmu_budget["งบประมาณเฉลี่ย"] / 1e6

display(
    pmu_budget[["จำนวน", "ร้อยละโครงการ", "งบเฉลี่ย (ล้านบาท)"]].round(2)
)

# count x size
pmu_count_size = pd.crosstab(
    pmu_long["ประเภทตามกรอบ_บพท."],
    pmu_long[size_std_col]
).reindex(columns=size_order, fill_value=0)

# budget mean x size
pmu_budget_size = (
    pmu_long.groupby(["ประเภทตามกรอบ_บพท.", size_std_col])[budget_col]
      .mean()
      .unstack()
      .reindex(columns=size_order)
      / 1e6
)

# เรียงตาม pmu_table
pmu_count_size = pmu_count_size.reindex(pmu_table.index)
pmu_budget_size = pmu_budget_size.reindex(pmu_table.index)

# plot stacked bar
plot_stacked_count_with_budget(
    count_table=pmu_count_size,
    budget_table=pmu_budget_size,
    total_pct_table=pmu_table[["จำนวน", "ร้อยละ"]],
    title="ประเภทของการวิจัยตามกรอบ บพท.",
    filename="4_3_2_pmu_research_framework.png",
    label_map=PMU_LABELS,
    wrap_width=20,
    figsize=(15, 8),
    rotate_xticks=25,
    note="หมายเหตุ: Multiple response — 1 โครงการอาจอยู่ได้มากกว่า 1 ประเภท"
)

# 4.3 การดำเนินงานของแผนงานวิจัย

## 4.3.1 หน่วยงานผู้รับทุน
ส่วนนี้ Word ระบุให้เขียนบรรยาย จึงใช้ตารางสังกัดจาก 4.2.3 เป็นข้อมูลประกอบ

## 4.3.2 ประเภทของงานวิจัย
วิเคราะห์ทั้งประเภทการวิจัยหลัก และประเภทการวิจัยตามกรอบ บพท. แบบ multiple response


In [ ]:
research_type_col = "ประเภทของการวิจัย"
research_type = frequency_table(df[research_type_col], denom=N)
display(research_type[["จำนวน", "ร้อยละ", "n (%)"]])

type_budget = (
    df.groupby(research_type_col)
      .agg(
          n_projects=("รหัสโครงการ", "count"),
          budget_mean=("งบประมาณที่ได้รับจัดสรร", "mean"),
          budget_total=("งบประมาณที่ได้รับจัดสรร", "sum")
      )
      .rename(columns={"n_projects":"จำนวน", "budget_mean":"งบประมาณเฉลี่ย", "budget_total":"งบประมาณรวม"})
)
type_budget["ร้อยละ"] = type_budget["จำนวน"]/N*100
type_budget["งบเฉลี่ย (ล้านบาท)"] = type_budget["งบประมาณเฉลี่ย"]/1e6
display(type_budget[["จำนวน", "ร้อยละ", "งบเฉลี่ย (ล้านบาท)"]].round(2))

# ============================================================
# กราฟ 4.3.2 (1) ประเภทของการวิจัย
# Bar = จำนวนโครงการ
# จุด = งบประมาณเฉลี่ย แยกตามขนาด เล็ก / กลาง / ใหญ่
# ============================================================

plot_df = research_type[["จำนวน", "ร้อยละ"]].copy()

budget_by_size = (
    df.groupby(
        [research_type_col, "ขนาดโครงการ"]
    )["งบประมาณที่ได้รับจัดสรร"]
    .mean()
    .unstack()
    / 1e6
)

budget_by_size = budget_by_size.reindex(plot_df.index)

x = np.arange(len(plot_df))

fig, ax1 = plt.subplots(figsize=(11, 7))

bar_colors = [
    "#8FB9E0",
    "#A8D5BA",
    "#F6C28B"
]

bars = ax1.bar(
    x,
    plot_df["จำนวน"].values,
    color=bar_colors[:len(plot_df)],
    edgecolor="white",
    linewidth=1,
    width=0.62
)

ax1.set_title(
    "ประเภทของการวิจัย",
    fontsize=17,
    fontweight="bold",
    pad=16
)

ax1.set_ylabel("จำนวนโครงการ", fontsize=13)
ax1.set_xlabel("")

ax1.set_xticks(x)
ax1.set_xticklabels(
    plot_df.index,
    fontsize=11
)

ax1.grid(
    axis="y",
    linestyle="--",
    alpha=0.18
)

ax1.set_axisbelow(True)
ax1.spines[["top"]].set_visible(False)

max_count = max(plot_df["จำนวน"].max(), 1)

for bar, (_, row) in zip(
    bars,
    plot_df.iterrows()
):
    ax1.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + max_count*0.02,
        f'{int(row["จำนวน"])} ({row["ร้อยละ"]:.1f}%)',
        ha="center",
        va="bottom",
        fontsize=11
    )

ax1.set_ylim(
    0,
    max_count * 1.22
)

# ------------------------------------------------------------
# แกน Y ขวา = งบประมาณเฉลี่ย
# ------------------------------------------------------------

ax2 = ax1.twinx()

ax2.set_ylabel(
    "งบประมาณเฉลี่ย (ล้านบาท/โครงการ)",
    fontsize=13
)

offsets = {
    "เล็ก": -0.16,
    "กลาง": 0,
    "ใหญ่": 0.16
}

markers = {
    "เล็ก": "o",
    "กลาง": "s",
    "ใหญ่": "^"
}

marker_colors = {
    "เล็ก": "#6BAED6",
    "กลาง": "#74C476",
    "ใหญ่": "#FD8D3C"
}

for size in ["เล็ก", "กลาง", "ใหญ่"]:

    if size not in budget_by_size.columns:
        continue

    values = budget_by_size[size]
    valid = values.notna()

    xx = x[valid] + offsets[size]
    yy = values[valid].values

    ax2.scatter(
        xx,
        yy,
        s=90,
        marker=markers[size],
        color=marker_colors[size],
        edgecolor="white",
        linewidth=1,
        label=f"ขนาด{size}",
        zorder=5
    )

    for xpos, val in zip(xx, yy):
        ax2.annotate(
            f"{val:.2f}",
            (xpos, val),
            xytext=(0, 8),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=9,
            color=marker_colors[size]
        )

ax2.legend(
    loc="upper right",
    frameon=False,
    fontsize=10,
    title="งบเฉลี่ย"
)

ax2.spines["top"].set_visible(False)

fig.tight_layout()

fig.savefig(
    OUTPUT_DIR / "4_3_2_research_type.png",
    dpi=CHART_STYLE.get("save_dpi", 240),
    bbox_inches="tight"
)

plt.show()

# ประเภทการวิจัย x ขนาดโครงการ
cross_type_size = pd.crosstab(
    df[research_type_col].map(clean_text),
    df["ขนาดโครงการ"].map(clean_text),
    margins=True
)
display(cross_type_size)

# ประเภทงานวิจัยตามกรอบ บพท. เป็น multiple response
pmu_cols = [
    "ประเภทของการวิจัยตามกรอบ บพท. (1) (Gift)",
    "ประเภทของการวิจัยตามกรอบ (2) บพท. (Gift)",
    "ประเภทของการวิจัยตามกรอบ (3) (Gift)"
]
pmu_long = (
    df[["รหัสโครงการ", "ขนาดโครงการ", "งบประมาณที่ได้รับจัดสรร"] + pmu_cols]
      .melt(
          id_vars=["รหัสโครงการ", "ขนาดโครงการ", "งบประมาณที่ได้รับจัดสรร"],
          value_vars=pmu_cols,
          value_name="ประเภทตามกรอบ_บพท."
      )
      .dropna(subset=["ประเภทตามกรอบ_บพท."])
)
pmu_long["ประเภทตามกรอบ_บพท."] = pmu_long["ประเภทตามกรอบ_บพท."].map(clean_text)
pmu_long = pmu_long.drop_duplicates(["รหัสโครงการ", "ประเภทตามกรอบ_บพท."])

pmu_count = pmu_long["ประเภทตามกรอบ_บพท."].value_counts()
pmu_table = pd.DataFrame({"จำนวน": pmu_count})
pmu_table["ร้อยละ"] = pmu_table["จำนวน"]/N*100
pmu_table["n (%)"] = [n_pct(n, N) for n in pmu_table["จำนวน"]]
display(pmu_table)

# ============================================================
# กราฟ 4.3.2 (2) ประเภทของการวิจัยตามกรอบ บพท.
# Bar = จำนวนโครงการ
# จุด = งบประมาณเฉลี่ย แยกตามขนาด เล็ก / กลาง / ใหญ่
# ============================================================

plot_df = pmu_table[["จำนวน", "ร้อยละ"]].copy()

pmu_budget_size = (
    pmu_long.groupby(
        ["ประเภทตามกรอบ_บพท.", "ขนาดโครงการ"]
    )["งบประมาณที่ได้รับจัดสรร"]
    .mean()
    .unstack()
    / 1e6
)

pmu_budget_size = pmu_budget_size.reindex(plot_df.index)

display_labels = apply_display_labels(
    plot_df.index,
    label_map=PMU_LABELS,
    wrap_width=22
)

x = np.arange(len(plot_df))

fig, ax1 = plt.subplots(figsize=(14, 8))

bar_colors = [
    "#8FB9E0",
    "#A8D5BA",
    "#F6C28B",
    "#C7B6E5",
    "#F2B5B5",
    "#9ED9CC",
    "#F7D6A3"
]

bars = ax1.bar(
    x,
    plot_df["จำนวน"].values,
    color=bar_colors[:len(plot_df)],
    edgecolor="white",
    linewidth=1,
    width=0.62
)

ax1.set_title(
    "ประเภทของการวิจัยตามกรอบ บพท.",
    fontsize=17,
    fontweight="bold",
    pad=16
)

ax1.set_ylabel("จำนวนโครงการ", fontsize=13)
ax1.set_xlabel("")

ax1.set_xticks(x)
ax1.set_xticklabels(
    display_labels,
    fontsize=10,
    rotation=25,
    ha="right"
)

ax1.grid(
    axis="y",
    linestyle="--",
    alpha=0.18
)

ax1.set_axisbelow(True)
ax1.spines[["top"]].set_visible(False)

max_count = max(plot_df["จำนวน"].max(), 1)

for bar, (_, row) in zip(
    bars,
    plot_df.iterrows()
):
    ax1.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + max_count*0.02,
        f'{int(row["จำนวน"])} ({row["ร้อยละ"]:.1f}%)',
        ha="center",
        va="bottom",
        fontsize=10
    )

ax1.set_ylim(
    0,
    max_count * 1.25
)

# ------------------------------------------------------------
# แกน Y ขวา = งบประมาณเฉลี่ย
# ------------------------------------------------------------

ax2 = ax1.twinx()

ax2.set_ylabel(
    "งบประมาณเฉลี่ย (ล้านบาท/โครงการ)",
    fontsize=13
)

offsets = {
    "เล็ก": -0.16,
    "กลาง": 0,
    "ใหญ่": 0.16
}

markers = {
    "เล็ก": "o",
    "กลาง": "s",
    "ใหญ่": "^"
}

marker_colors = {
    "เล็ก": "#6BAED6",
    "กลาง": "#74C476",
    "ใหญ่": "#FD8D3C"
}

for size in ["เล็ก", "กลาง", "ใหญ่"]:

    if size not in pmu_budget_size.columns:
        continue

    values = pmu_budget_size[size]
    valid = values.notna()

    xx = x[valid] + offsets[size]
    yy = values[valid].values

    ax2.scatter(
        xx,
        yy,
        s=90,
        marker=markers[size],
        color=marker_colors[size],
        edgecolor="white",
        linewidth=1,
        label=f"ขนาด{size}",
        zorder=5
    )

    for xpos, val in zip(xx, yy):
        ax2.annotate(
            f"{val:.2f}",
            (xpos, val),
            xytext=(0, 8),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=9,
            color=marker_colors[size]
        )

ax2.legend(
    loc="upper right",
    frameon=False,
    fontsize=10,
    title="งบเฉลี่ย"
)

ax2.spines["top"].set_visible(False)

fig.text(
    0.01,
    0.01,
    "หมายเหตุ: Multiple response — 1 โครงการอาจอยู่ได้มากกว่า 1 ประเภท",
    fontsize=9
)

fig.tight_layout(
    rect=[0, 0.04, 1, 1]
)

fig.savefig(
    OUTPUT_DIR / "4_3_2_pmu_research_framework.png",
    dpi=CHART_STYLE.get("save_dpi", 240),
    bbox_inches="tight"
)

plt.show()

pmu_budget = (
    pmu_long.groupby("ประเภทตามกรอบ_บพท.")
      .agg(
          n_projects=("รหัสโครงการ", "nunique"),
          budget_mean=("งบประมาณที่ได้รับจัดสรร", "mean")
      )
      .rename(columns={"n_projects":"จำนวน", "budget_mean":"งบประมาณเฉลี่ย"})
)
pmu_budget["ร้อยละโครงการ"] = pmu_budget["จำนวน"]/N*100
pmu_budget["งบเฉลี่ย (ล้านบาท)"] = pmu_budget["งบประมาณเฉลี่ย"]/1e6
display(pmu_budget[["จำนวน", "ร้อยละโครงการ", "งบเฉลี่ย (ล้านบาท)"]].round(2))


## 4.3.3 สาขางานวิจัย (OECD)

ใช้ `สาขาการวิจัย (OECD1)` เป็นสาขาหลักตามกรอบ Word และแสดงจำนวน/ร้อยละกำกับทุกแท่ง


In [ ]:
oecd_field = frequency_table(df["สาขาการวิจัย (OECD1)"], denom=N)
display(oecd_field[["จำนวน", "ร้อยละ", "n (%)"]])
save_table(oecd_field, "4_3_3_oecd_research_field.csv")

barh_count(
    oecd_field[["จำนวน", "ร้อยละ"]],
    "สาขาการวิจัยหลักของโครงการ (OECD)",
    filename="4_3_3_oecd_research_field.png",
    label_map=OECD_FIELD_LABELS,
    wrap_width=28,
    color=COLORS["primary"]
)


# 4.4 ผลผลิตของแผนงานวิจัย

## 4.4.1 ประเภทของผลผลิต

หนึ่งโครงการอาจมีผลผลิตได้หลายประเภท จึงแปลงผลผลิตลำดับที่ 1–6 เป็น long format ก่อนนับ  
**ข้อควรระวัง:** หากนำงบประมาณโครงการไปผูกกับผลผลิตหลายประเภท งบประมาณจะถูกนับซ้ำเมื่อรวมข้ามประเภท ดังนั้นตารางนี้เหมาะสำหรับเปรียบเทียบ “งบของโครงการที่มีผลผลิตประเภทนั้น” ไม่ควรนำยอดข้ามประเภทมาบวกเป็นงบรวมแผนงาน


In [ ]:
output_cols = [
    "ผลผลิตหลัก ลำดับที่ 1",
    "ผลผลิตหลัก ลำดับที่ 2",
    "ผลผลิตหลัก ลำดับที่ 3",
    "ผลผลิตหลัก ลำดับที่ 4",
    "ผลผลิตหลัก ลำดับที่ 5",
    "ผลผลิตหลัก ลำดับที่ 6",
]

output_long = (
    df[["รหัสโครงการ", research_type_col, "งบประมาณที่ได้รับจัดสรร"] + output_cols]
      .melt(
          id_vars=["รหัสโครงการ", research_type_col, "งบประมาณที่ได้รับจัดสรร"],
          value_vars=output_cols,
          value_name="ประเภทผลผลิต"
      )
      .dropna(subset=["ประเภทผลผลิต"])
)
output_long["ประเภทผลผลิต"] = output_long["ประเภทผลผลิต"].map(clean_text)
output_long = output_long.drop_duplicates(["รหัสโครงการ", "ประเภทผลผลิต"])

output_count = output_long["ประเภทผลผลิต"].value_counts()
output_table = pd.DataFrame({"จำนวน": output_count})
output_table["ร้อยละ"] = output_table["จำนวน"]/N*100
output_table["n (%)"] = [n_pct(n, N) for n in output_table["จำนวน"]]

display(output_table)
save_table(output_table, "4_4_1_output_types.csv")

barh_count(
    output_table[["จำนวน", "ร้อยละ"]],
    "ประเภทผลผลิตของโครงการ (Multiple response)",
    filename="4_4_1_output_types.png",
    label_map=OUTPUT_LABELS,
    wrap_width=30,
    color=COLORS["primary"],
    figsize=(11, max(6, 0.62*len(output_table)+2)),
    note="หมายเหตุ: Multiple response — 1 โครงการอาจมีผลผลิตมากกว่า 1 ประเภท"
)

# ผลผลิต x ประเภทการวิจัย
output_by_type = pd.crosstab(
    output_long["ประเภทผลผลิต"],
    output_long[research_type_col]
)
display(output_by_type)

# งบประมาณของโครงการที่มีผลผลิตแต่ละประเภท
output_budget = (
    output_long.groupby("ประเภทผลผลิต")
      .agg(
          n_projects=("รหัสโครงการ", "nunique"),
          budget_total=("งบประมาณที่ได้รับจัดสรร", "sum"),
          budget_mean=("งบประมาณที่ได้รับจัดสรร", "mean")
      )
      .rename(columns={
          "n_projects":"จำนวนโครงการ",
          "budget_total":"งบประมาณรวมของโครงการ",
          "budget_mean":"งบประมาณเฉลี่ยต่อโครงการ"
      })
)
output_budget["งบรวมของโครงการ (ล้านบาท)"] = output_budget["งบประมาณรวมของโครงการ"]/1e6
output_budget["งบเฉลี่ยต่อโครงการ (ล้านบาท)"] = output_budget["งบประมาณเฉลี่ยต่อโครงการ"]/1e6
display(
    output_budget[
        ["จำนวนโครงการ", "งบรวมของโครงการ (ล้านบาท)", "งบเฉลี่ยต่อโครงการ (ล้านบาท)"]
    ].round(2)
)


In [ ]:
# ============================================================
# HELPER: Full Pie Chart
# ใช้สี pastel โทนเดิม
# ไม่มีรูตรงกลาง
# ไม่มี N ตรงกลาง
# ============================================================

import matplotlib.pyplot as plt

PIE_PASTEL_COLORS = [
    "#8FB9E0",  # ฟ้าอ่อน
    "#A8D5BA",  # เขียวอ่อน
    "#F6C28B",  # พีชอ่อน
    "#C7B6E5",  # ม่วงอ่อน
    "#F2B5B5",  # ชมพูอ่อน
    "#9ED9CC",  # มิ้นต์อ่อน
    "#F7D6A3",  # ครีมส้ม
    "#AFCBFF",  # ฟ้าอ่อน 2
    "#F4B6C2",  # โรส
    "#B7D88C",  # เขียวอ่อน 2
    "#D9C2F0",  # lavender
    "#FFD6A5",  # apricot
]

def autopct_with_n(values):
    total = sum(values)

    def _fmt(pct):
        n = int(round(pct * total / 100.0))
        return f"{pct:.1f}%\n(n={n})"

    return _fmt


def plot_pie_from_table(
    table,
    title,
    filename,
    note=None,
    label_map=None,
    figsize=(9, 7)
):
    """
    table ต้องมี:
    - index = category
    - column 'จำนวน'
    - optional column 'ร้อยละ'
    """

    plot_df = table.copy()

    # ตัดแถวที่จำนวน = 0
    plot_df = plot_df[plot_df["จำนวน"] > 0].copy()

    labels = []
    for idx in plot_df.index:
        idx_str = str(idx)

        if label_map and idx_str in label_map:
            labels.append(label_map[idx_str])
        else:
            labels.append(idx_str)

    values = plot_df["จำนวน"].astype(float).tolist()

    colors = [
        PIE_PASTEL_COLORS[i % len(PIE_PASTEL_COLORS)]
        for i in range(len(values))
    ]

    fig, ax = plt.subplots(figsize=figsize)

    wedges, texts, autotexts = ax.pie(
        values,
        labels=labels,
        autopct=autopct_with_n(values),
        startangle=90,
        counterclock=False,
        colors=colors,
        pctdistance=0.68,
        labeldistance=1.08,
        wedgeprops=dict(edgecolor="white", linewidth=1.2)
    )

    for t in texts:
        t.set_fontsize(10)

    for t in autotexts:
        t.set_fontsize(10)
        t.set_color("#333333")

    ax.set_title(
        title,
        fontsize=17,
        fontweight="bold",
        pad=16
    )

    if note:
        fig.text(
            0.01, 0.01,
            note,
            ha="left",
            va="bottom",
            fontsize=9,
            color="#555555"
        )
        fig.tight_layout(rect=[0, 0.05, 1, 1])
    else:
        fig.tight_layout()

    fig.savefig(
        OUTPUT_DIR / filename,
        dpi=CHART_STYLE.get("save_dpi", 240),
        bbox_inches="tight"
    )

    plt.show()

## 4.4.2 TRL ณ สิ้นสุดโครงการ

กราฟใช้ระดับ TRL เรียงจากต่ำไปสูง ไม่เรียงตามจำนวน เพื่อให้เห็น “ระดับความพร้อม” เป็นลำดับต่อเนื่อง


In [ ]:
trl = numeric_prefix(df["TRL สิ้นสุด (Gift)"])
trl_label = trl.map(lambda x: f"TRL {int(x)}" if pd.notna(x) else "N/A")
trl_table = frequency_table(trl_label, denom=N, sort=False)

trl_order = [f"TRL {i}" for i in range(1,10)] + ["N/A"]
trl_table = trl_table.reindex([x for x in trl_order if x in trl_table.index])

# เพิ่มงบ
trl_budget = (
    df.assign(TRL_label=trl_label)
      .groupby("TRL_label", dropna=False)["งบประมาณที่ได้รับจัดสรร"]
      .sum()
)
trl_table["งบประมาณรวม"] = trl_budget.reindex(trl_table.index)
trl_table["สัดส่วนงบประมาณ"] = trl_table["งบประมาณรวม"]/budget.sum()*100
trl_table["งบรวม (ล้านบาท)"] = trl_table["งบประมาณรวม"]/1e6

display(trl_table[["จำนวน", "ร้อยละ", "งบรวม (ล้านบาท)", "สัดส่วนงบประมาณ"]].round(2))
save_table(trl_table, "4_4_2_trl.csv")

# ============================================================
# 4.4.2 TRL
# Pie / Donut chart
# ============================================================

# ถ้าใน cell เดิมคุณมี trl_table อยู่แล้ว ใช้ต่อได้เลย
# trl_table ต้องมี column: จำนวน, ร้อยละ
# และ index เป็นระดับ TRL

display(
    trl_table[
        ["จำนวน", "ร้อยละ", "n (%)"]
    ]
)

save_table(
    trl_table,
    "4_4_2_trl.csv"
)

TRL_LABELS = {
    "TRL 1": "TRL 1",
    "TRL 2": "TRL 2",
    "TRL 3": "TRL 3",
    "TRL 4": "TRL 4",
    "TRL 5": "TRL 5",
    "TRL 6": "TRL 6",
    "TRL 7": "TRL 7",
    "TRL 8": "TRL 8",
    "TRL 9": "TRL 9",
}

plot_pie_from_table(
    table=trl_table[["จำนวน", "ร้อยละ"]],
    title="ระดับความพร้อมทางเทคโนโลยี (TRL)",
    filename="4_4_2_trl.png",
    label_map=TRL_LABELS,
    note="หมายเหตุ: ตัวเลขในกราฟแสดงร้อยละและจำนวนโครงการในแต่ละระดับ TRL"
)


In [ ]:
# ============================================================
# 4.4.2.1 - 4.4.2.3
# เพิ่ม 3 ภาพต่อจาก 4.4.2
#
# 4_4_2_1 : TRL จำแนกตามประเภทของการวิจัย
# 4_4_2_2 : TRL จำแนกตามประเภทผลผลิตของโครงการ
# 4_4_2_3 : TRL จำแนกตามจำนวนประเภทผลผลิตของโครงการ (Bubble)
# ============================================================

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import textwrap
from pathlib import Path


# ============================================================
# 0) FALLBACKS / HELPERS
# ============================================================

# OUTPUT_DIR fallback
if "OUTPUT_DIR" not in globals():
    OUTPUT_DIR = Path("chapter4_outputs")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# dpi fallback
SAVE_DPI = 240
if "CHART_STYLE" in globals():
    SAVE_DPI = CHART_STYLE.get("save_dpi", 240)

# save_table fallback
def save_table_local(table, filename):
    out = OUTPUT_DIR / filename
    table.to_csv(out, encoding="utf-8-sig")
    return out

# clean text fallback
def clean_text_local(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    x = re.sub(r"\s+", " ", x)
    return x if x != "" else np.nan

# wrap text
def wrap_label(text, width=22):
    if pd.isna(text):
        return ""
    text = str(text)
    return "\n".join(
        textwrap.wrap(
            text,
            width=width,
            break_long_words=False,
            break_on_hyphens=False
        )
    )

# detect output columns
def detect_output_cols(columns):
    cols = [c for c in columns if "ผลผลิตหลัก" in str(c)]
    def _extract_num(s):
        m = re.search(r"(\d+)", str(s))
        return int(m.group(1)) if m else 999
    return sorted(cols, key=_extract_num)

# detect TRL column
def detect_trl_col(columns):
    candidates = []

    preferred_names = [
        "TRL",
        "ระดับความพร้อมทางเทคโนโลยี (TRL)",
        "ระดับความพร้อมทางเทคโนโลยี",
        "Technology Readiness Level (TRL)"
    ]

    for p in preferred_names:
        for c in columns:
            if str(c).strip() == p:
                return c

    for c in columns:
        c_upper = str(c).upper()
        if "TRL" in c_upper:
            candidates.append(c)

    if len(candidates) > 0:
        # เอาตัวที่ชื่อสั้นสุดก่อน
        candidates = sorted(candidates, key=lambda x: len(str(x)))
        return candidates[0]

    raise ValueError("ไม่พบคอลัมน์ TRL ใน df กรุณาตรวจชื่อคอลัมน์อีกครั้ง")

# normalize TRL
def normalize_trl(x):
    if pd.isna(x):
        return "N/A"

    s = str(x).strip().upper()

    if s in ["N/A", "NA", "NONE", "-", "ไม่ระบุ", "ยังไม่ระบุ", ""]:
        return "N/A"

    # หาเลข 1-9
    m = re.search(r"([1-9])", s)
    if m:
        return f"TRL {m.group(1)}"

    return "N/A"


# ============================================================
# 1) SETTINGS
# ============================================================

project_id_col = "รหัสโครงการ"
research_type_col = "ประเภทของการวิจัย"
budget_col = "งบประมาณที่ได้รับจัดสรร"

output_cols = detect_output_cols(df.columns)
trl_col = detect_trl_col(df.columns)

print("Detected output columns:", output_cols)
print("Detected TRL column:", trl_col)

N_projects = df[project_id_col].nunique()
print("จำนวนโครงการทั้งหมด =", N_projects)

trl_order = [f"TRL {i}" for i in range(1, 10)] + ["N/A"]

trl_colors = {
    "TRL 1": "#DCEBFA",
    "TRL 2": "#CBE0F7",
    "TRL 3": "#B9D5F2",
    "TRL 4": "#A8D5BA",
    "TRL 5": "#BEE3C8",
    "TRL 6": "#F6D7A8",
    "TRL 7": "#F6C28B",
    "TRL 8": "#E9B2D0",
    "TRL 9": "#C7B6E5",
    "N/A"  : "#D9D9D9"
}

# label map สำหรับ output
OUTPUT_DISPLAY = {
    "ฐานข้อมูล ระบบและกลไก": "ฐานข้อมูล\nระบบและกลไก",
    "กำลังคน หรือหน่วยงาน ที่ได้รับการพัฒนาทักษะ": "กำลังคน/หน่วยงาน\nที่ได้รับการพัฒนาทักษะ",
    "นวัตกรรมทางสังคม": "นวัตกรรม\nทางสังคม",
    "เครือข่าย": "เครือข่าย",
    "อื่นๆ": "อื่นๆ",
    "ข้อเสนอแนะเชิงนโยบาย:ข้อเสนอที่มุ่งใช้ประกอบการตัดสินใจ การกำหนดนโยบาย มาตรการ แผนงาน แนวทางปฏิบัติ หรือกฎเกณฑ์ของหน่วยงานหรือองค์กร":
        "ข้อเสนอแนะเชิงนโยบาย\n(รายการ 1)",
    "ข้อเสนอแนะเชิงนโยบาย":
        "ข้อเสนอแนะเชิงนโยบาย\n(รายการ 2)",
    "ต้นแบบผลิตภัณฑ์":
        "ต้นแบบผลิตภัณฑ์",
    "เครื่องมือ และโครงสร้างพื้นฐานที่สร้างขึ้น หรือพัฒนาต่อยอดภายใต้โครงการ":
        "เครื่องมือ/\nโครงสร้างพื้นฐาน",
    "เทคโนโลยี/กระบวนการใหม่":
        "เทคโนโลยี/\nกระบวนการใหม่"
}

def output_display_label(x):
    x = str(x)
    if x in OUTPUT_DISPLAY:
        return OUTPUT_DISPLAY[x]
    return wrap_label(x, width=18)

def research_display_label(x):
    return wrap_label(x, width=20)


# ============================================================
# 2) PREPARE BASE DF
# ============================================================

base_df = df.copy()
base_df["TRL_มาตรฐาน"] = base_df[trl_col].apply(normalize_trl)

# project-level dedup
project_df = (
    base_df[
        [project_id_col, research_type_col, budget_col, "TRL_มาตรฐาน"]
    ]
    .drop_duplicates(subset=[project_id_col])
    .copy()
)

# output_long สร้างใหม่เพื่อให้ cell นี้รันเดี่ยวได้
output_long = (
    base_df[
        [project_id_col, budget_col, "TRL_มาตรฐาน"] + output_cols
    ]
    .melt(
        id_vars=[project_id_col, budget_col, "TRL_มาตรฐาน"],
        value_vars=output_cols,
        value_name="ประเภทผลผลิต"
    )
    .dropna(subset=["ประเภทผลผลิต"])
)

output_long["ประเภทผลผลิต"] = output_long["ประเภทผลผลิต"].apply(clean_text_local)

# โครงการเดียวกัน + output เดียวกัน นับครั้งเดียว
output_long = (
    output_long
    .drop_duplicates(subset=[project_id_col, "ประเภทผลผลิต"])
    .copy()
)


# ============================================================
# 3) HELPER: STACKED BAR FOR TRL
# ============================================================

def plot_stacked_trl(
    count_table,
    total_series,
    pct_series,
    title,
    filename,
    x_label,
    display_func=None,
    note=None,
    figsize=(14, 8),
    rotate_xticks=0,
    csv_filename=None
):
    """
    count_table: index = category, cols = TRL 1..9 / N/A
    total_series: unique total per category
    pct_series: percentage of total projects
    """

    plot_df = count_table.copy()

    # เรียง columns ให้ตรง order
    plot_df = plot_df.reindex(columns=trl_order, fill_value=0)

    if csv_filename is not None:
        save_table_local(plot_df, csv_filename)

    x = np.arange(len(plot_df))
    fig, ax = plt.subplots(figsize=figsize)

    bottom = np.zeros(len(plot_df))

    for trl in trl_order:
        values = plot_df[trl].fillna(0).astype(float).values

        bars = ax.bar(
            x,
            values,
            bottom=bottom,
            width=0.68,
            color=trl_colors[trl],
            edgecolor="white",
            linewidth=1.0,
            label=trl
        )

        # label ในชั้น
        for i, bar in enumerate(bars):
            n = values[i]
            if n <= 0:
                continue

            y_center = bottom[i] + n / 2

            if n >= 2:
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    y_center,
                    f"{int(n)}",
                    ha="center",
                    va="center",
                    fontsize=8.5,
                    fontweight="bold",
                    color="#333333"
                )
            elif n == 1:
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    y_center,
                    "1",
                    ha="center",
                    va="center",
                    fontsize=8,
                    fontweight="bold",
                    color="#333333"
                )

        bottom = bottom + values

    max_total = max(float(bottom.max()), 1)

    # label บนยอดแท่ง
    for i, idx in enumerate(plot_df.index):
        total_n = int(total_series.loc[idx])
        total_pct = float(pct_series.loc[idx])

        ax.text(
            x[i],
            bottom[i] + max_total * 0.03,
            f"{total_n} ({total_pct:.1f}%)",
            ha="center",
            va="bottom",
            fontsize=9.5,
            fontweight="bold",
            color="#333333"
        )

    # x labels
    if display_func is not None:
        labels = [display_func(idx) for idx in plot_df.index]
    else:
        labels = [str(idx) for idx in plot_df.index]

    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=10)

    if rotate_xticks != 0:
        plt.setp(ax.get_xticklabels(), rotation=rotate_xticks, ha="right")

    ax.set_title(
        title,
        fontsize=18,
        fontweight="bold",
        pad=16
    )
    ax.set_ylabel("จำนวนโครงการ", fontsize=13)
    ax.set_xlabel(x_label, fontsize=12)

    ax.grid(axis="y", linestyle="--", alpha=0.18)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)

    ax.set_ylim(0, max_total * 1.25)

    ax.legend(
        title="ระดับ TRL",
        bbox_to_anchor=(1.01, 1),
        loc="upper left",
        frameon=False,
        fontsize=9,
        title_fontsize=10
    )

    if note:
        fig.text(
            0.01, 0.01,
            note,
            ha="left",
            va="bottom",
            fontsize=9,
            color="#555555"
        )
        fig.tight_layout(rect=[0, 0.05, 0.82, 1])
    else:
        fig.tight_layout(rect=[0, 0, 0.82, 1])

    fig.savefig(
        OUTPUT_DIR / filename,
        dpi=SAVE_DPI,
        bbox_inches="tight"
    )

    plt.show()


# ============================================================
# 4) 4_4_2_1 : TRL จำแนกตามประเภทการวิจัย
# ============================================================

research_trl_count = pd.crosstab(
    project_df[research_type_col],
    project_df["TRL_มาตรฐาน"]
).reindex(columns=trl_order, fill_value=0)

research_total = project_df.groupby(research_type_col)[project_id_col].nunique()
research_pct = research_total / N_projects * 100

# เรียงตามจำนวนรวม
research_order = research_total.sort_values(ascending=False).index
research_trl_count = research_trl_count.reindex(research_order)
research_total = research_total.reindex(research_order)
research_pct = research_pct.reindex(research_order)

display(research_trl_count)
save_table_local(research_trl_count, "4_4_2_1_trl_by_research_type.csv")

plot_stacked_trl(
    count_table=research_trl_count,
    total_series=research_total,
    pct_series=research_pct,
    title="ระดับความพร้อมทางเทคโนโลยี (TRL)\nจำแนกตามประเภทของการวิจัย",
    filename="4_4_2_1_trl_by_research_type.png",
    x_label="ประเภทของการวิจัย",
    display_func=research_display_label,
    note="หมายเหตุ: ตัวเลขในชั้นของแท่งแสดงจำนวนโครงการในแต่ละระดับ TRL และตัวเลขบนยอดแท่งแสดงจำนวนโครงการรวมของประเภทการวิจัยนั้น",
    figsize=(13, 8),
    rotate_xticks=0,
    csv_filename=None
)


# ============================================================
# 5) 4_4_2_2 : TRL จำแนกตามประเภทผลผลิต
# ============================================================

output_trl_count = pd.crosstab(
    output_long["ประเภทผลผลิต"],
    output_long["TRL_มาตรฐาน"]
).reindex(columns=trl_order, fill_value=0)

output_total = output_long.groupby("ประเภทผลผลิต")[project_id_col].nunique()
output_pct = output_total / N_projects * 100

# เรียงตามจำนวนรวม
output_order = output_total.sort_values(ascending=False).index
output_trl_count = output_trl_count.reindex(output_order)
output_total = output_total.reindex(output_order)
output_pct = output_pct.reindex(output_order)

display(output_trl_count)
save_table_local(output_trl_count, "4_4_2_2_trl_by_output_type.csv")

plot_stacked_trl(
    count_table=output_trl_count,
    total_series=output_total,
    pct_series=output_pct,
    title="ระดับความพร้อมทางเทคโนโลยี (TRL)\nจำแนกตามประเภทผลผลิตของโครงการ",
    filename="4_4_2_2_trl_by_output_type.png",
    x_label="ประเภทผลผลิต",
    display_func=output_display_label,
    note="หมายเหตุ: ประเภทผลผลิตเป็น Multiple response; 1 โครงการอาจมีผลผลิตมากกว่า 1 ประเภท แต่ในผลผลิตแต่ละประเภทจะนับโครงการซ้ำเพียงครั้งเดียว",
    figsize=(18, 9),
    rotate_xticks=25,
    csv_filename=None
)


# ============================================================
# 4_4_2_3 : Bubble chart
# TRL จำแนกตามจำนวนประเภทผลผลิตของโครงการ
#
# X = TRL
# Y = จำนวนประเภทผลผลิตต่อโครงการ
# Bubble size = งบประมาณรวม
# ตัวเลขใน Bubble = จำนวนโครงการ
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# 1. จำนวน "ประเภทผลผลิต" ต่อโครงการ
# ============================================================

project_output_count = (
    output_long
    .groupby(project_id_col)["ประเภทผลผลิต"]
    .nunique()
    .reset_index()
)

project_output_count = project_output_count.rename(
    columns={
        "ประเภทผลผลิต": "จำนวนประเภทผลผลิต"
    }
)


# ============================================================
# 2. เตรียมข้อมูลระดับโครงการ
# ============================================================

bubble_project = (
    project_df[
        [
            project_id_col,
            budget_col,
            "TRL_มาตรฐาน"
        ]
    ]
    .copy()
)


bubble_project = bubble_project.merge(
    project_output_count,
    on=project_id_col,
    how="left"
)


# โครงการไม่มี output ที่ระบุ ให้เป็น 0
bubble_project[
    "จำนวนประเภทผลผลิต"
] = (
    bubble_project[
        "จำนวนประเภทผลผลิต"
    ]
    .fillna(0)
    .astype(int)
)


# ============================================================
# 3. แปลง TRL เป็นตำแหน่งตัวเลขบนแกน X
# ============================================================

trl_to_num = {
    "N/A": 0,
    "TRL 1": 1,
    "TRL 2": 2,
    "TRL 3": 3,
    "TRL 4": 4,
    "TRL 5": 5,
    "TRL 6": 6,
    "TRL 7": 7,
    "TRL 8": 8,
    "TRL 9": 9
}


bubble_project[
    "TRL_num"
] = (
    bubble_project[
        "TRL_มาตรฐาน"
    ]
    .map(trl_to_num)
)


# ถ้ามีค่า TRL ที่ map ไม่ได้
# ให้เป็น 0 = N/A
bubble_project[
    "TRL_num"
] = (
    bubble_project[
        "TRL_num"
    ]
    .fillna(0)
    .astype(int)
)


# ============================================================
# 4. จำนวนโครงการในแต่ละจุด
#
# ใช้ size() โดยตรง
# ป้องกัน KeyError จาก named aggregation
# ============================================================

bubble_count = (
    bubble_project
    .groupby(
        [
            "TRL_มาตรฐาน",
            "TRL_num",
            "จำนวนประเภทผลผลิต"
        ]
    )
    .size()
    .reset_index(
        name="จำนวนโครงการ"
    )
)


# ============================================================
# 5. งบประมาณรวม / เฉลี่ย ในแต่ละจุด
# ============================================================

bubble_budget = (
    bubble_project
    .groupby(
        [
            "TRL_มาตรฐาน",
            "TRL_num",
            "จำนวนประเภทผลผลิต"
        ],
        as_index=False
    )[budget_col]
    .agg(
        [
            "sum",
            "mean"
        ]
    )
    .reset_index()
)


bubble_budget = bubble_budget.rename(
    columns={
        "sum": "งบประมาณรวม",
        "mean": "งบประมาณเฉลี่ย"
    }
)


# ============================================================
# 6. Merge จำนวนโครงการ + งบประมาณ
# ============================================================

bubble_cell = bubble_count.merge(
    bubble_budget,
    on=[
        "TRL_มาตรฐาน",
        "TRL_num",
        "จำนวนประเภทผลผลิต"
    ],
    how="left"
)


# ============================================================
# 7. แปลงงบเป็นล้านบาท
# ============================================================

bubble_cell[
    "งบรวม (ล้านบาท)"
] = (
    bubble_cell[
        "งบประมาณรวม"
    ]
    / 1_000_000
)


bubble_cell[
    "งบเฉลี่ย (ล้านบาท)"
] = (
    bubble_cell[
        "งบประมาณเฉลี่ย"
    ]
    / 1_000_000
)


# ============================================================
# 8. ตรวจสอบก่อน Plot
# ============================================================

print(
    "Columns bubble_cell:"
)

print(
    bubble_cell.columns.tolist()
)

print()

display(
    bubble_cell.sort_values(
        [
            "TRL_num",
            "จำนวนประเภทผลผลิต"
        ]
    )
)


# ============================================================
# 9. Save ตาราง
# ============================================================

bubble_export = (
    bubble_cell
    .sort_values(
        [
            "TRL_num",
            "จำนวนประเภทผลผลิต"
        ]
    )
    .copy()
)


bubble_export.to_csv(
    OUTPUT_DIR / "4_4_2_3_trl_vs_num_outputs.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 10. คำนวณขนาด Bubble
#
# Bubble size = งบประมาณรวม
# แต่ normalize เพื่อไม่ให้วงใหญ่กินทั้งภาพ
# ============================================================

budget_values = (
    bubble_cell[
        "งบรวม (ล้านบาท)"
    ]
    .fillna(0)
    .to_numpy(
        dtype=float
    )
)


positive_budget = (
    budget_values[
        budget_values > 0
    ]
)


MIN_BUBBLE = 600
MAX_BUBBLE = 3500


if len(positive_budget) == 0:

    bubble_sizes = np.repeat(
        1200,
        len(bubble_cell)
    )

else:

    b_min = positive_budget.min()
    b_max = positive_budget.max()

    if b_min == b_max:

        bubble_sizes = np.repeat(
            1800,
            len(bubble_cell)
        )

    else:

        bubble_sizes = (
            MIN_BUBBLE
            +
            (
                (budget_values - b_min)
                /
                (b_max - b_min)
            )
            *
            (
                MAX_BUBBLE
                - MIN_BUBBLE
            )
        )

        bubble_sizes = np.clip(
            bubble_sizes,
            MIN_BUBBLE,
            MAX_BUBBLE
        )


# ============================================================
# 11. สีตาม TRL
# ============================================================

trl_colors_local = {
    "TRL 1": "#DCEBFA",
    "TRL 2": "#CBE0F7",
    "TRL 3": "#B9D5F2",
    "TRL 4": "#A8D5BA",
    "TRL 5": "#BEE3C8",
    "TRL 6": "#F6D7A8",
    "TRL 7": "#F6C28B",
    "TRL 8": "#E9B2D0",
    "TRL 9": "#C7B6E5",
    "N/A":   "#D9D9D9"
}


bubble_colors = [
    trl_colors_local.get(
        trl,
        "#D9D9D9"
    )
    for trl
    in bubble_cell[
        "TRL_มาตรฐาน"
    ]
]


# ============================================================
# 12. Plot
# ============================================================

fig, ax = plt.subplots(
    figsize=(15, 9)
)


ax.scatter(
    bubble_cell[
        "TRL_num"
    ],

    bubble_cell[
        "จำนวนประเภทผลผลิต"
    ],

    s=bubble_sizes,

    c=bubble_colors,

    alpha=0.82,

    edgecolors="#666666",

    linewidth=1.2,

    zorder=3
)


# ============================================================
# 13. จำนวนโครงการกลางวง
# ============================================================

for _, row in bubble_cell.iterrows():

    ax.text(
        row["TRL_num"],
        row["จำนวนประเภทผลผลิต"],

        str(
            int(
                row["จำนวนโครงการ"]
            )
        ),

        ha="center",
        va="center",

        fontsize=10,
        fontweight="bold",

        color="#333333",

        zorder=4
    )


# ============================================================
# 14. Annotation
#
# แสดงเฉพาะกรณีมีมากกว่า 1 โครงการ
# หรืองบเฉลี่ยเพื่อช่วยอ่านค่า
# ============================================================

for _, row in bubble_cell.iterrows():

    label_text = (
        f'{int(row["จำนวนโครงการ"])} โครงการ\n'
        f'งบเฉลี่ย '
        f'{row["งบเฉลี่ย (ล้านบาท)"]:.2f} ลบ.'
    )

    ax.annotate(
        label_text,

        xy=(
            row["TRL_num"],
            row["จำนวนประเภทผลผลิต"]
        ),

        xytext=(
            16,
            12
        ),

        textcoords="offset points",

        ha="left",
        va="bottom",

        fontsize=8.3,

        color="#555555",

        bbox=dict(
            boxstyle="round,pad=0.20",
            facecolor="white",
            edgecolor="#DDDDDD",
            linewidth=0.5,
            alpha=0.85
        ),

        zorder=5
    )


# ============================================================
# 15. X AXIS = TRL
# ============================================================

ax.set_xticks(
    range(0, 10)
)

ax.set_xticklabels(
    [
        "N/A",
        "TRL 1",
        "TRL 2",
        "TRL 3",
        "TRL 4",
        "TRL 5",
        "TRL 6",
        "TRL 7",
        "TRL 8",
        "TRL 9"
    ],
    fontsize=10
)


ax.set_xlabel(
    "ระดับความพร้อมทางเทคโนโลยี (TRL)",
    fontsize=13
)


# ============================================================
# 16. Y AXIS = จำนวนประเภทผลผลิต
# ============================================================

max_outputs = max(
    int(
        bubble_project[
            "จำนวนประเภทผลผลิต"
        ]
        .max()
    ),
    1
)


ax.set_yticks(
    range(
        0,
        max_outputs + 1
    )
)


ax.set_ylabel(
    "จำนวนประเภทผลผลิตของโครงการ",
    fontsize=13
)


# ============================================================
# 17. Title
# ============================================================

ax.set_title(
    "ระดับความพร้อมทางเทคโนโลยี (TRL)\n"
    "จำแนกตามจำนวนประเภทผลผลิตของโครงการ",
    fontsize=18,
    fontweight="bold",
    pad=18
)


# ============================================================
# 18. Layout / limits
# ============================================================

ax.set_xlim(
    -0.65,
    9.65
)


ax.set_ylim(
    -0.5,
    max_outputs + 1.3
)


ax.grid(
    linestyle="--",
    alpha=0.18
)

ax.set_axisbelow(
    True
)


ax.spines[
    ["top", "right"]
].set_visible(
    False
)


# ============================================================
# 19. Note
# ============================================================

fig.text(
    0.01,
    0.01,

    (
        "หมายเหตุ: ตัวเลขในวงกลม = จำนวนโครงการ; "
        "ขนาดวงกลม = งบประมาณรวมของโครงการในจุดนั้น; "
        "แกน Y = จำนวนประเภทผลผลิตที่พบในแต่ละโครงการ"
    ),

    fontsize=9,

    color="#555555"
)


fig.tight_layout(
    rect=[
        0,
        0.05,
        1,
        1
    ]
)


# ============================================================
# 20. Save
# ============================================================

filename = (
    "4_4_2_3_trl_vs_num_outputs.png"
)


fig.savefig(
    OUTPUT_DIR / filename,
    dpi=SAVE_DPI,
    bbox_inches="tight"
)


plt.show()


print(
    "สร้างไฟล์เรียบร้อย:",
    filename
)
print("- 4_4_2_1_trl_by_research_type.png")
print("- 4_4_2_2_trl_by_output_type.png")
print("- 4_4_2_3_trl_vs_num_outputs.png")

## 4.4.3 SRL ณ สิ้นสุดโครงการ

ใช้หลักเดียวกับ TRL โดยเรียงระดับ SRL จากต่ำไปสูง และคง N/A แยกต่างหาก


In [ ]:
srl = numeric_prefix(df["SRL สิ้นสุด(Gift)"])
srl_label = srl.map(lambda x: f"SRL {int(x)}" if pd.notna(x) else "N/A")
srl_table = frequency_table(srl_label, denom=N, sort=False)

srl_order = [f"SRL {i}" for i in range(1,10)] + ["N/A"]
srl_table = srl_table.reindex([x for x in srl_order if x in srl_table.index])

srl_budget = (
    df.assign(SRL_label=srl_label)
      .groupby("SRL_label", dropna=False)["งบประมาณที่ได้รับจัดสรร"]
      .sum()
)
srl_table["งบประมาณรวม"] = srl_budget.reindex(srl_table.index)
srl_table["สัดส่วนงบประมาณ"] = srl_table["งบประมาณรวม"]/budget.sum()*100
srl_table["งบรวม (ล้านบาท)"] = srl_table["งบประมาณรวม"]/1e6

display(srl_table[["จำนวน", "ร้อยละ", "งบรวม (ล้านบาท)", "สัดส่วนงบประมาณ"]].round(2))
save_table(srl_table, "4_4_3_srl.csv")

# ============================================================
# 4.4.3 SRL
# Pie / Donut chart
# ============================================================

# ถ้าใน cell เดิมคุณมี srl_table อยู่แล้ว ใช้ต่อได้เลย
# srl_table ต้องมี column: จำนวน, ร้อยละ
# และ index เป็นระดับ SRL

display(
    srl_table[
        ["จำนวน", "ร้อยละ", "n (%)"]
    ]
)

save_table(
    srl_table,
    "4_4_3_srl.csv"
)

SRL_LABELS = {
    "SRL 1": "SRL 1",
    "SRL 2": "SRL 2",
    "SRL 3": "SRL 3",
    "SRL 4": "SRL 4",
    "SRL 5": "SRL 5",
    "SRL 6": "SRL 6",
    "SRL 7": "SRL 7",
    "SRL 8": "SRL 8",
    "SRL 9": "SRL 9",
}

plot_pie_from_table(
    table=srl_table[["จำนวน", "ร้อยละ"]],
    title="ระดับความพร้อมทางสังคม (SRL)",
    filename="4_4_3_srl.png",
    label_map=SRL_LABELS,
    note="หมายเหตุ: ตัวเลขในกราฟแสดงร้อยละและจำนวนโครงการในแต่ละระดับ SRL"
)

# 4.5 ผลประโยชน์ทางวิชาการ

## 4.5.1 ผลประโยชน์ระดับผลผลิต

รวมจำนวนบทความ การเผยแพร่ ตำรา/คู่มือ สื่อ และการใช้ประโยชน์ด้านการเรียนการสอนตามคอลัมน์ในฐานข้อมูล  
ส่วน “การพัฒนาบุคลากร” มีเพียงตัวแปรที่ฐานข้อมูลรองรับ จึงไม่เติมตัวเลขจากแหล่งอื่น


In [ ]:
academic_cols = [
    "จำนวนบทความ (ระดับนานาชาติ)",
    "จำนวนร่างบทความ (ระดับนานาชาติ)",
    "จำนวนบทความ (ระดับประเทศ)",
    "จำนวนร่างบทความ (ระดับประเทศ)",
    "จำนวนบทความ (นำเสนอในที่ประชุมระดับนานาชาติ)",
    "จำนวนบทความ (นำเสนอในที่ประชุมระดับประเทศ)",
    "จำนวน ตำรา/หนังสือ (จำนวนเรื่อง)",
    "จำนวนคู่มือ/สิ่งพิมพ์",
    "จำนวนครั้งการเผยแพร่ผ่านการอบรม/สัมมนา/เวทีสาธารณะ/นิทรรศการ",
    "จำนวนสื่อ clip vdo หรือเพจเผยแพร่/งานเขียนออนไลน์",
    "จำนวนครั้งการเผยแพร่ผ่าน วิดีทัศน์ โทรทัศน์ วิทยุ นสพ. อินเตอร์เน็ต",
    "มีการใช้ประโยชน์กับการเรียนการสอน (จำนวนวิชา)",
    "มีการใช้ประโยชน์กับการวิจัยเพื่อพัฒนานิสิต (จำนวนนิสิต ที่ทำวิจัยหรือวิทยานิพนธ์)",
]

acad_sum = {}
for c in academic_cols:
    acad_sum[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).sum()

academic_table = (
    pd.Series(acad_sum, name="จำนวนรวม")
      .sort_values(ascending=False)
      .to_frame()
)
display(academic_table)
save_table(academic_table, "4_5_1_academic_benefits.csv")

# ============================================================
# 4.5.1 ผลประโยชน์ทางวิชาการระดับผลผลิต
# กราฟแท่งหลายสี pastel
# ============================================================

plot_acad = academic_table[
    academic_table["จำนวนรวม"] > 0
].copy()

if not plot_acad.empty:

    plot_acad = plot_acad.sort_values(
        "จำนวนรวม",
        ascending=False
    )

    display_labels = apply_display_labels(
        plot_acad.index,
        label_map=ACADEMIC_LABELS,
        wrap_width=24
    )

    n_cat = len(plot_acad)

    # --------------------------------------------------------
    # สี pastel โทนเดียวกับกราฟอื่น
    # --------------------------------------------------------

    academic_palette = [
        "#8FB9E0",  # ฟ้า
        "#A8D5BA",  # เขียว
        "#F6C28B",  # พีช
        "#C7B6E5",  # ม่วง
        "#F2B5B5",  # ชมพู
        "#9ED9CC",  # มิ้นต์
        "#F7D6A3",  # ครีม
        "#AFCBFF",  # ฟ้าอ่อน
        "#F4B6C2",  # โรส
        "#B7D88C",  # เขียวอ่อน
        "#D9C2F0",  # lavender
        "#FFD6A5",  # apricot
        "#B8D8D8",  # blue-green
    ]

    bar_colors = [
        academic_palette[
            i % len(academic_palette)
        ]
        for i in range(n_cat)
    ]

    # --------------------------------------------------------
    # Figure
    # --------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=(
            max(12, n_cat * 1.15),
            8.2
        )
    )

    bars = ax.bar(
        range(n_cat),
        plot_acad["จำนวนรวม"].values,
        color=bar_colors,
        edgecolor="white",
        linewidth=0.8
    )

    # --------------------------------------------------------
    # Title / Axis
    # --------------------------------------------------------

    ax.set_title(
        "ผลประโยชน์ทางวิชาการระดับผลผลิต",
        weight="bold",
        pad=14,
        fontsize=CHART_STYLE["title_size"]
    )

    ax.set_ylabel(
        "จำนวนรวม",
        fontsize=CHART_STYLE["axis_label_size"]
    )

    ax.set_xlabel("")

    ax.set_xticks(
        range(n_cat)
    )

    ax.set_xticklabels(
        display_labels,
        fontsize=CHART_STYLE["tick_size"]
    )

    plt.setp(
        ax.get_xticklabels(),
        rotation=30,
        ha="right"
    )

    ax.tick_params(
        axis="y",
        labelsize=CHART_STYLE["tick_size"]
    )

    # --------------------------------------------------------
    # Grid / style
    # --------------------------------------------------------

    ax.grid(
        axis="y",
        alpha=CHART_STYLE["grid_alpha"],
        linestyle="--"
    )

    ax.set_axisbelow(True)

    ax.spines[
        ["top", "right"]
    ].set_visible(False)

    # --------------------------------------------------------
    # Y limit
    # --------------------------------------------------------

    maxv = max(
        plot_acad["จำนวนรวม"].max(),
        1
    )

    ax.set_ylim(
        0,
        maxv * 1.18 + 1
    )

    # --------------------------------------------------------
    # ตัวเลขบนแท่ง
    # --------------------------------------------------------

    for rect, value in zip(
        bars,
        plot_acad["จำนวนรวม"].values
    ):

        ax.annotate(
            f"{int(value)}",

            xy=(
                rect.get_x()
                + rect.get_width() / 2,

                rect.get_height()
            ),

            xytext=(
                0,
                5
            ),

            textcoords="offset points",

            ha="center",
            va="bottom",

            fontsize=CHART_STYLE[
                "annotation_size"
            ],

            color=COLORS["dark"]
        )

    # --------------------------------------------------------
    # Layout / Save
    # --------------------------------------------------------

    fig.tight_layout()

    fig.savefig(
        OUTPUT_DIR / "4_5_1_academic_benefits.png",
        dpi=CHART_STYLE["save_dpi"],
        bbox_inches="tight"
    )

    plt.show()

# ทรัพย์สินทางปัญญา
patent_status = pd.to_numeric(
    df["สิทธิบัตร สถานะ 0= ไม่มี, 1=ได้แล้ว2=รออนุมัติ 3=กำลังจะจด"],
    errors="coerce"
)
petty_patent_status = pd.to_numeric(
    df["อนุสิทธิบัตร สถานะ 0= ไม่มี, 1=ได้แล้ว2=รออนุมัติ 3=กำลังจะจด"],
    errors="coerce"
)

ip_summary = pd.DataFrame({
    "สิทธิบัตร": patent_status.value_counts().sort_index(),
    "อนุสิทธิบัตร": petty_patent_status.value_counts().sort_index(),
}).fillna(0).astype(int)

ip_summary.index = [
    {0:"0 = ไม่มี", 1:"1 = ได้แล้ว", 2:"2 = รออนุมัติ", 3:"3 = กำลังจะจด"}.get(int(i), str(i))
    for i in ip_summary.index
]
display(ip_summary)



## 4.5.2 ผลประโยชน์ทางวิชาการระดับผลลัพธ์

เอกสาร Word ต้องการ “การได้รับอ้างอิงบทความทางวิชาการระดับนานาชาติ” แต่ในชีต `สถานภาพแผนงาน -Clean` **ไม่พบตัวแปรจำนวนการอ้างอิงโดยตรง**  
จึงไม่คำนวณแทนด้วยตัวแปรอื่น เพื่อไม่ให้ความหมายคลาดเคลื่อน


# 4.6 ผลลัพธ์ (Outcome) และผลกระทบ (Impacts)

## 4.6.1.1 กลุ่มผู้ใช้ประโยชน์

ผู้ใช้ประโยชน์เป็น multiple response โดยรวมคอลัมน์ผู้ใช้ประโยชน์ที่ 1–5  
จากนั้นวิเคราะห์ซ้ำเฉพาะโครงการที่ฐานข้อมูลระบุว่า “เกิดผลลัพธ์แล้ว”


In [ ]:
user_cols = [f"ผู้ใช้ประโยชน์ที่ {i}" for i in range(1,6)]

user_long = (
    df[["รหัสโครงการ", "เกิดผลลัพธ์ แล้วหรือไม่"] + user_cols]
      .melt(
          id_vars=["รหัสโครงการ", "เกิดผลลัพธ์ แล้วหรือไม่"],
          value_vars=user_cols,
          value_name="กลุ่มผู้ใช้ประโยชน์"
      )
      .dropna(subset=["กลุ่มผู้ใช้ประโยชน์"])
)
user_long["กลุ่มผู้ใช้ประโยชน์"] = user_long["กลุ่มผู้ใช้ประโยชน์"].map(clean_text)
user_long = user_long.drop_duplicates(["รหัสโครงการ", "กลุ่มผู้ใช้ประโยชน์"])

user_count = user_long["กลุ่มผู้ใช้ประโยชน์"].value_counts()
user_table = pd.DataFrame({"จำนวน": user_count})
user_table["ร้อยละ"] = user_table["จำนวน"]/N*100
user_table["n (%)"] = [n_pct(n, N) for n in user_table["จำนวน"]]

display(user_table)
barh_count(
    user_table[["จำนวน","ร้อยละ"]],
    "กลุ่มผู้ใช้ประโยชน์เป้าหมาย (Multiple response)",
    filename="4_6_1_1_target_users.png",
    label_map=USER_LABELS,
    wrap_width=28,
    color=COLORS["secondary"],
    note="หมายเหตุ: Multiple response — 1 โครงการอาจมีกลุ่มผู้ใช้มากกว่า 1 กลุ่ม"
)

# เกิดผลลัพธ์แล้ว: รองรับทั้งเลข 1 และข้อความที่ขึ้นต้นด้วย 1
outcome_happened = numeric_prefix(df["เกิดผลลัพธ์ แล้วหรือไม่"]).eq(1)
actual_outcome_ids = set(df.loc[outcome_happened, "รหัสโครงการ"])

actual_user_long = user_long[user_long["รหัสโครงการ"].isin(actual_outcome_ids)]
actual_user_count = actual_user_long["กลุ่มผู้ใช้ประโยชน์"].value_counts()

actual_user_table = pd.DataFrame({"จำนวน": actual_user_count})
actual_denom = len(actual_outcome_ids)
actual_user_table["ร้อยละ"] = actual_user_table["จำนวน"]/actual_denom*100 if actual_denom else 0
actual_user_table["n (%)"] = [n_pct(n, actual_denom) for n in actual_user_table["จำนวน"]]

print("จำนวนโครงการที่ระบุว่าเกิดผลลัพธ์แล้ว:", actual_denom)
display(actual_user_table)

if not actual_user_table.empty:
    barh_count(
        actual_user_table[["จำนวน","ร้อยละ"]],
        "กลุ่มผู้ใช้ประโยชน์ในโครงการที่ระบุว่าเกิดผลลัพธ์แล้ว",
        filename="4_6_1_1_actual_users.png",
        label_map=USER_LABELS,
        wrap_width=28,
        color=COLORS["secondary"],
        note="หมายเหตุ: Multiple response — คำนวณจากโครงการที่ระบุว่าเกิดผลลัพธ์แล้ว"
    )


## 4.6.1.2 ประเภทผลลัพธ์ของงานวิจัย

รวมผลลัพธ์ที่ 1–5 เป็น multiple response และสามารถ cross-tab ตามขนาดโครงการ ประเภทการวิจัย และกลุ่มผู้ใช้ประโยชน์ได้


In [ ]:
# ============================================================
# 4.6.1.2 ประเภทผลลัพธ์ของโครงการ
#
# Multiple response
# 1 โครงการอาจมีผลลัพธ์มากกว่า 1 ประเภท
#
# IMPORTANT:
# รวมหมวด
# "เพิ่มผลิตภาพการผลิต/เพิ่มผลผลิต"
# "เพิ่มผลิตภาพการผลิต/เพิ่มผลผลิต และ"
# "เพิ่มผลิตภาพการผลิต/เพิ่มผลผลิต/เพิ่มประสิทธิภาพการดำเนินงาน"
#
# ให้เป็นหมวดเดียวกันก่อนนับ
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import textwrap
from pathlib import Path


# ============================================================
# 0. SETTINGS
# ============================================================

project_id_col = "รหัสโครงการ"

outcome_cols = [
    "ผลลัพธ์ที่ 1",
    "ผลลัพธ์ที่ 2",
    "ผลลัพธ์ที่ 3",
    "ผลลัพธ์ที่ 4",
    "ผลลัพธ์ที่ 5",
]

N_projects = df[project_id_col].nunique()

print(
    "จำนวนโครงการทั้งหมด =",
    N_projects
)


# ============================================================
# 1. HELPER
# ============================================================

def clean_outcome_text(x):

    if pd.isna(x):
        return np.nan

    x = str(x).strip()

    # รวม whitespace ซ้ำ
    x = " ".join(
        x.split()
    )

    if x == "":
        return np.nan

    return x


# ============================================================
# 2. ชื่อมาตรฐานของ Outcome
# ============================================================

PRODUCTIVITY_CANONICAL = (
    "เพิ่มผลิตภาพการผลิต/"
    "เพิ่มผลผลิต/"
    "เพิ่มประสิทธิภาพการดำเนินงาน"
)


# ============================================================
# 3. RECODE MAP
#
# ใส่หลายรูปแบบไว้เผื่อมีความต่างเรื่องคำ/ช่องว่าง
# ============================================================

OUTCOME_RECODE = {

    # --------------------------------------------------------
    # กลุ่ม Productivity / Output / Efficiency
    # --------------------------------------------------------

    "เพิ่มผลิตภาพการผลิต/เพิ่มผลผลิต":
        PRODUCTIVITY_CANONICAL,

    "เพิ่มผลิตภาพการผลิต/เพิ่มผลผลิต และ":
        PRODUCTIVITY_CANONICAL,

    "เพิ่มผลิตภาพการผลิต/เพิ่มผลผลิต/เพิ่มประสิทธิภาพการดำเนินงาน":
        PRODUCTIVITY_CANONICAL,


    # --------------------------------------------------------
    # กลุ่มลดความสูญเสีย
    # รวม wording สั้นกับ wording รายละเอียด
    # --------------------------------------------------------

    "ลดความสูญเสียทางเศรษฐกิจและสังคม":
        "ลดความสูญเสียทางเศรษฐกิจและสังคม",

    (
        "ลดความสูญเสียทางเศรษฐกิจสังคม "
        "เช่น ลดอัตราการเจ็บป่วย ลดอัตราการตาย "
        "และลดค่าใช้จ่ายด้านสุขภาพ เป็นต้น"
    ):
        "ลดความสูญเสียทางเศรษฐกิจและสังคม",


    # --------------------------------------------------------
    # คุณภาพชีวิต
    # --------------------------------------------------------

    "พัฒนาคุณภาพชีวิต":
        "พัฒนาคุณภาพชีวิต",

    "พัฒนาคุณภาพชีวิต (เพราะเข้าถึงบริการสุขภาพคุณภาพมาตรฐาน)":
        "พัฒนาคุณภาพชีวิต",
}


# ============================================================
# 4. WIDE -> LONG
# ============================================================

outcome_long = (
    df[
        [project_id_col]
        + outcome_cols
    ]
    .melt(
        id_vars=[
            project_id_col
        ],
        value_vars=outcome_cols,
        value_name="ประเภทผลลัพธ์"
    )
    .dropna(
        subset=[
            "ประเภทผลลัพธ์"
        ]
    )
)


# ============================================================
# 5. CLEAN TEXT
# ============================================================

outcome_long[
    "ประเภทผลลัพธ์"
] = (
    outcome_long[
        "ประเภทผลลัพธ์"
    ]
    .apply(
        clean_outcome_text
    )
)


outcome_long = (
    outcome_long
    .dropna(
        subset=[
            "ประเภทผลลัพธ์"
        ]
    )
)


# ============================================================
# 6. NORMALIZE / RECODE
# ============================================================

outcome_long[
    "ประเภทผลลัพธ์"
] = (
    outcome_long[
        "ประเภทผลลัพธ์"
    ]
    .replace(
        OUTCOME_RECODE
    )
)


# ============================================================
# 7. ป้องกันการนับซ้ำ
#
# ตัวอย่าง:
# ถ้าโครงการเดียวมีทั้ง
# "เพิ่มผลิตภาพการผลิต/เพิ่มผลผลิต"
# และ
# "เพิ่มผลิตภาพ.../เพิ่มประสิทธิภาพ..."
#
# หลังรวมชื่อแล้ว ให้นับเพียง 1 ครั้ง
# ============================================================

outcome_long = (
    outcome_long
    .drop_duplicates(
        subset=[
            project_id_col,
            "ประเภทผลลัพธ์"
        ]
    )
    .copy()
)


# ============================================================
# 8. ตารางความถี่
# ============================================================

outcome_count = (
    outcome_long[
        "ประเภทผลลัพธ์"
    ]
    .value_counts()
)


outcome_table = pd.DataFrame(
    {
        "จำนวน": outcome_count
    }
)


outcome_table[
    "ร้อยละ"
] = (
    outcome_table[
        "จำนวน"
    ]
    / N_projects
    * 100
)


outcome_table[
    "n (%)"
] = [
    f"{int(n)} ({n / N_projects * 100:.1f}%)"
    for n
    in outcome_table[
        "จำนวน"
    ]
]


# ============================================================
# 9. เรียงจากมาก -> น้อย
# ============================================================

outcome_table = (
    outcome_table
    .sort_values(
        "จำนวน",
        ascending=False
    )
)


# ============================================================
# 10. DISPLAY TABLE
# ============================================================

display(
    outcome_table
)


# ============================================================
# 11. SAVE TABLE
# ============================================================

if "save_table" in globals():

    save_table(
        outcome_table,
        "4_6_1_2_outcomes.csv"
    )

else:

    outcome_table.to_csv(
        OUTPUT_DIR /
        "4_6_1_2_outcomes.csv",
        encoding="utf-8-sig"
    )


# ============================================================
# 12. เตรียมข้อมูลสำหรับ Plot
#
# N/A แยกออกจาก Outcome จริง
# ============================================================

plot_outcome = (
    outcome_table[
        outcome_table.index.astype(str).str.upper()
        != "N/A"
    ]
    .copy()
)


# ============================================================
# 13. DISPLAY LABELS
# ============================================================

OUTCOME_DISPLAY = {

    PRODUCTIVITY_CANONICAL:
        "เพิ่มผลิตภาพการผลิต/\n"
        "เพิ่มผลผลิต/\n"
        "เพิ่มประสิทธิภาพการดำเนินงาน",

    "สร้างเครือข่ายและความร่วมมือทางสังคม":
        "สร้างเครือข่ายและ\n"
        "ความร่วมมือทางสังคม",

    "ลดความสูญเสียทางเศรษฐกิจและสังคม":
        "ลดความสูญเสียทาง\n"
        "เศรษฐกิจและสังคม",

    "พัฒนาคุณภาพชีวิต":
        "พัฒนาคุณภาพชีวิต",

    "พัฒนาชุมชน":
        "พัฒนาชุมชน",

    "เพิ่มรายได้":
        "เพิ่มรายได้",

    "ลดต้นทุนการผลิต":
        "ลดต้นทุนการผลิต",

    "ลดขยะและของเสีย":
        "ลดขยะและของเสีย",

    "คุณภาพสิ่งแวดล้อมดีขึ้น ลดมลพิษ/มลภาวะ/ลดก๊าซเรือนกระจก":
        "คุณภาพสิ่งแวดล้อมดีขึ้น/\n"
        "ลดมลพิษและก๊าซเรือนกระจก",
}


def outcome_display_label(x):

    x = str(x)

    if x in OUTCOME_DISPLAY:
        return OUTCOME_DISPLAY[x]

    return "\n".join(
        textwrap.wrap(
            x,
            width=24,
            break_long_words=False,
            break_on_hyphens=False
        )
    )


display_labels = [
    outcome_display_label(x)
    for x
    in plot_outcome.index
]


# ============================================================
# 14. Pastel colors
# โทนเดียวกับกราฟอื่นในบท
# ============================================================

outcome_palette = [
    "#8FB9E0",
    "#A8D5BA",
    "#F6C28B",
    "#C7B6E5",
    "#F2B5B5",
    "#9ED9CC",
    "#F7D6A3",
    "#AFCBFF",
    "#F4B6C2",
    "#B7D88C",
    "#D9C2F0",
    "#FFD6A5",
]


bar_colors = [
    outcome_palette[
        i % len(outcome_palette)
    ]
    for i in range(
        len(plot_outcome)
    )
]


# ============================================================
# 15. PLOT
# ============================================================

n_cat = len(
    plot_outcome
)


fig, ax = plt.subplots(
    figsize=(
        max(
            14,
            n_cat * 1.35
        ),
        8.5
    )
)


bars = ax.bar(
    range(n_cat),

    plot_outcome[
        "จำนวน"
    ].values,

    color=bar_colors,

    edgecolor="white",

    linewidth=0.9
)


# ============================================================
# 16. ตัวเลขบนแท่ง
# ============================================================

max_value = max(
    plot_outcome[
        "จำนวน"
    ].max(),
    1
)


for i, (
    rect,
    (_, row)
) in enumerate(
    zip(
        bars,
        plot_outcome.iterrows()
    )
):

    n = int(
        row[
            "จำนวน"
        ]
    )

    pct = float(
        row[
            "ร้อยละ"
        ]
    )

    ax.annotate(
        f"{n} ({pct:.1f}%)",

        xy=(
            rect.get_x()
            + rect.get_width()/2,

            rect.get_height()
        ),

        xytext=(
            0,
            5
        ),

        textcoords="offset points",

        ha="center",
        va="bottom",

        fontsize=10,

        fontweight="bold",

        color="#333333"
    )


# ============================================================
# 17. AXIS
# ============================================================

ax.set_xticks(
    range(
        n_cat
    )
)


ax.set_xticklabels(
    display_labels,
    fontsize=10
)


plt.setp(
    ax.get_xticklabels(),
    rotation=28,
    ha="right"
)


ax.set_ylabel(
    "จำนวนโครงการ",
    fontsize=13
)


ax.set_xlabel(
    "ประเภทผลลัพธ์",
    fontsize=12
)


# ============================================================
# 18. TITLE
# ============================================================

ax.set_title(
    "ประเภทผลลัพธ์ของโครงการ",
    fontsize=18,
    fontweight="bold",
    pad=16
)


# ============================================================
# 19. STYLE
# ============================================================

ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.18
)


ax.set_axisbelow(
    True
)


ax.spines[
    ["top", "right"]
].set_visible(
    False
)


ax.set_ylim(
    0,
    max_value * 1.20 + 1
)


# ============================================================
# 20. NOTE
# ============================================================

fig.text(
    0.01,
    0.01,

    (
        "หมายเหตุ: Multiple response — "
        "1 โครงการอาจมีผลลัพธ์มากกว่า 1 ประเภท; "
        "กลุ่ม 'เพิ่มผลิตภาพการผลิต/เพิ่มผลผลิต' "
        "ถูกรวมกับ 'เพิ่มผลิตภาพการผลิต/เพิ่มผลผลิต/"
        "เพิ่มประสิทธิภาพการดำเนินงาน' ก่อนการนับ "
        "และโครงการเดียวกันในหมวดเดียวกันนับเพียง 1 ครั้ง"
    ),

    fontsize=9,

    color="#555555"
)


# ============================================================
# 21. LAYOUT
# ============================================================

fig.tight_layout(
    rect=[
        0,
        0.06,
        1,
        1
    ]
)


# ============================================================
# 22. SAVE GRAPH
# ============================================================

fig.savefig(
    OUTPUT_DIR /
    "4_6_1_2_outcomes.png",

    dpi=(
        CHART_STYLE.get(
            "save_dpi",
            240
        )
        if "CHART_STYLE" in globals()
        else 240
    ),

    bbox_inches="tight"
)


plt.show()


# ============================================================
# 23. ตรวจสอบ Outcome หลัง Normalize
# ============================================================

print(
    "\nผลการนับ Outcome หลังรวมหมวด:"
)

display(
    outcome_table[
        [
            "จำนวน",
            "ร้อยละ",
            "n (%)"
        ]
    ]
)


print(
    "\nจำนวนโครงการในหมวด Productivity ที่รวมแล้ว =",
    int(
        outcome_table.loc[
            PRODUCTIVITY_CANONICAL,
            "จำนวน"
        ]
    )
    if PRODUCTIVITY_CANONICAL
       in outcome_table.index
    else 0
)


## 4.6.2 ผลกระทบ

ฐานข้อมูลระบุ `0/1` ว่าโครงการมีผลกระทบ **ที่เกิดขึ้นหรือคาดว่าจะเกิดขึ้น** ในมิติเศรษฐกิจ สังคม และสิ่งแวดล้อม  
กราฟนี้จึงควรเขียนคำกำกับว่า **“เกิดขึ้น/คาดว่าจะเกิดขึ้น”** และไม่ตีความทุกค่า 1 ว่าเป็น Actual Impact


In [ ]:
impact_cols = {
    "เศรษฐกิจ": "ด้านเศรษฐกิจ",
    "สังคม": "ด้านสังคม",
    "สิ่งแวดล้อม": "ด้านสิ่งแวดล้อม"
}

impact_rows = []
for label, col in impact_cols.items():
    vals = pd.to_numeric(df[col], errors="coerce")
    n = vals.eq(1).sum()
    impact_rows.append([label, n, n/N*100])

impact_table = pd.DataFrame(
    impact_rows,
    columns=["มิติผลกระทบ", "จำนวน", "ร้อยละ"]
).set_index("มิติผลกระทบ")
impact_table["n (%)"] = [n_pct(n, N) for n in impact_table["จำนวน"]]

display(impact_table)
save_table(impact_table, "4_6_2_impact_dimensions.csv")

barh_count(
    impact_table[["จำนวน","ร้อยละ"]],
    "มิติผลกระทบที่เกิดขึ้น/คาดว่าจะเกิดขึ้น",
    filename="4_6_2_impact_dimensions.png",
    color=COLORS["accent"],
    wrap_width=24
)

# จำแนกผลกระทบตามขนาดโครงการและประเภทการวิจัย
for label, col in impact_cols.items():
    temp = df.loc[pd.to_numeric(df[col], errors="coerce").eq(1)]
    print(f"\n{label} — จำแนกตามขนาดโครงการ")
    display(temp["ขนาดโครงการ"].value_counts().to_frame("จำนวน"))
    print(f"{label} — จำแนกตามประเภทการวิจัย")
    display(temp[research_type_col].value_counts().to_frame("จำนวน"))


In [ ]:
# ============================================================
# 4.6.1.1 กลุ่มผู้ใช้ประโยชน์
# จำแนกตามประเภทของการวิจัย
# และประเภทการวิจัยตามกรอบ บพท.
#
# Output:
# 4_6_1_1_users_by_research_type.png
# 4_6_1_1_users_by_pmu_framework.png
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import textwrap


# ============================================================
# 0. SETTINGS
# ============================================================

project_id_col = "รหัสโครงการ"
research_type_col = "ประเภทของการวิจัย"

# ------------------------------------------------------------
# กลุ่มผู้ใช้ประโยชน์
# จากโครงสร้างข้อมูลเดิมมีสูงสุด 5 ช่อง
# ------------------------------------------------------------

user_cols = [
    "ผู้ใช้ประโยชน์ที่ 1",
    "ผู้ใช้ประโยชน์ที่ 2",
    "ผู้ใช้ประโยชน์ที่ 3",
    "ผู้ใช้ประโยชน์ที่ 4",
    "ผู้ใช้ประโยชน์ที่ 5",
]

# ------------------------------------------------------------
# ประเภทการวิจัยตามกรอบ บพท.
# ใช้ชื่อเดียวกับ 4.3.2
# ------------------------------------------------------------

pmu_cols = [
    "ประเภทของการวิจัยตามกรอบ บพท. (1) (Gift)",
    "ประเภทของการวิจัยตามกรอบ (2) บพท. (Gift)",
    "ประเภทของการวิจัยตามกรอบ (3) (Gift)"
]

N_projects = df[project_id_col].nunique()


# ============================================================
# 1. HELPER
# ============================================================

def clean_local_text(x):

    if pd.isna(x):
        return np.nan

    x = str(x).strip()

    if x == "":
        return np.nan

    return x


def wrap_local(text, width=20):

    if pd.isna(text):
        return ""

    return "\n".join(
        textwrap.wrap(
            str(text),
            width=width,
            break_long_words=False,
            break_on_hyphens=False
        )
    )


# ============================================================
# 2. USER LONG FORMAT
# ============================================================

user_long = (
    df[
        [project_id_col, research_type_col]
        + user_cols
    ]
    .melt(
        id_vars=[
            project_id_col,
            research_type_col
        ],
        value_vars=user_cols,
        value_name="กลุ่มผู้ใช้ประโยชน์"
    )
    .dropna(
        subset=[
            "กลุ่มผู้ใช้ประโยชน์"
        ]
    )
)


user_long[
    "กลุ่มผู้ใช้ประโยชน์"
] = (
    user_long[
        "กลุ่มผู้ใช้ประโยชน์"
    ]
    .apply(clean_local_text)
)


# โครงการเดียวกัน + ผู้ใช้ประเภทเดียวกัน
# นับเพียงครั้งเดียว
user_long = (
    user_long
    .drop_duplicates(
        [
            project_id_col,
            "กลุ่มผู้ใช้ประโยชน์"
        ]
    )
)


# ============================================================
# 3. USER DISPLAY LABELS
# ============================================================

USER_DISPLAY = {

    "หน่วยงานภาครัฐในระดับพื้นที่ ตำบล อำเภอ จังหวัด":
        "หน่วยงานรัฐ\nระดับพื้นที่",

    "ชุมชน/เครือข่าย":
        "ชุมชน/\nเครือข่าย",

    "ประชาชนกลุ่มเป้าหมาย":
        "ประชาชน\nกลุ่มเป้าหมาย",

    "โรงพยาบาล/สถานพยาบาล":
        "โรงพยาบาล/\nสถานพยาบาล",

    "บุคลากรทางการศึกษา/สถาบันการศึกษา":
        "บุคลากร/สถาบัน\nการศึกษา",

    "ผู้ป่วย":
        "ผู้ป่วย",

    "หน่วยงานภาครัฐระดับส่วนกลาง":
        "หน่วยงานรัฐ\nส่วนกลาง",

    "องค์กรปกครองส่วนท้องถิ่น":
        "องค์กรปกครอง\nส่วนท้องถิ่น",
}


def user_display_label(x):

    x = str(x)

    if x in USER_DISPLAY:
        return USER_DISPLAY[x]

    return wrap_local(
        x,
        width=18
    )


# ============================================================
# 4. COLOR PALETTE
# แต่ละกลุ่มผู้ใช้ = คนละสี pastel
# ============================================================

user_palette = [
    "#8FB9E0",
    "#A8D5BA",
    "#F6C28B",
    "#C7B6E5",
    "#F2B5B5",
    "#9ED9CC",
    "#F7D6A3",
    "#AFCBFF",
    "#F4B6C2",
    "#B7D88C",
    "#D9C2F0",
    "#FFD6A5",
]


# ============================================================
# 5. HELPER STACKED BAR
# ============================================================

def plot_user_stacked(
    count_table,
    unique_total,
    title,
    filename,
    x_label,
    x_label_func=None,
    rotate=0,
    figsize=(14, 8),
    note=None
):

    plot_df = count_table.copy()

    # เรียง user groups ตามจำนวนรวม
    user_order = (
        plot_df
        .sum(axis=0)
        .sort_values(
            ascending=False
        )
        .index
    )

    plot_df = plot_df[
        user_order
    ]

    x = np.arange(
        len(plot_df)
    )

    fig, ax = plt.subplots(
        figsize=figsize
    )

    bottom = np.zeros(
        len(plot_df)
    )

    colors = {
        col:
            user_palette[
                i % len(user_palette)
            ]
        for i, col
        in enumerate(
            plot_df.columns
        )
    }

    # --------------------------------------------------------
    # STACK
    # --------------------------------------------------------

    for user_group in plot_df.columns:

        values = (
            plot_df[
                user_group
            ]
            .fillna(0)
            .astype(float)
            .values
        )

        bars = ax.bar(
            x,
            values,
            bottom=bottom,
            width=0.68,
            color=colors[user_group],
            edgecolor="white",
            linewidth=1.0,
            label=user_display_label(
                user_group
            )
        )

        # จำนวนในแต่ละ layer
        for i, bar in enumerate(
            bars
        ):

            n = values[i]

            if n <= 0:
                continue

            y_center = (
                bottom[i]
                + n / 2
            )

            ax.text(
                bar.get_x()
                + bar.get_width()/2,
                y_center,
                str(int(n)),
                ha="center",
                va="center",
                fontsize=8.5,
                fontweight="bold",
                color="#333333"
            )

        bottom = bottom + values


    # --------------------------------------------------------
    # ยอดแท่ง = unique projects
    # --------------------------------------------------------

    max_stack = max(
        float(bottom.max()),
        1
    )

    for i, idx in enumerate(
        plot_df.index
    ):

        n_unique = int(
            unique_total.loc[idx]
        )

        pct = (
            n_unique
            / N_projects
            * 100
        )

        ax.text(
            x[i],
            bottom[i]
            + max_stack * 0.025,
            f"{n_unique} โครงการ\n({pct:.1f}%)",
            ha="center",
            va="bottom",
            fontsize=9,
            fontweight="bold",
            color="#333333"
        )


    # --------------------------------------------------------
    # X labels
    # --------------------------------------------------------

    if x_label_func:

        labels = [
            x_label_func(idx)
            for idx
            in plot_df.index
        ]

    else:

        labels = [
            wrap_local(
                idx,
                width=20
            )
            for idx
            in plot_df.index
        ]


    ax.set_xticks(x)

    ax.set_xticklabels(
        labels,
        fontsize=10
    )

    if rotate != 0:

        plt.setp(
            ax.get_xticklabels(),
            rotation=rotate,
            ha="right"
        )


    # --------------------------------------------------------
    # STYLE
    # --------------------------------------------------------

    ax.set_title(
        title,
        fontsize=18,
        fontweight="bold",
        pad=16
    )

    ax.set_ylabel(
        "จำนวนการจัดกลุ่มผู้ใช้ประโยชน์",
        fontsize=12
    )

    ax.set_xlabel(
        x_label,
        fontsize=12
    )

    ax.grid(
        axis="y",
        linestyle="--",
        alpha=0.18
    )

    ax.set_axisbelow(True)

    ax.spines[
        ["top", "right"]
    ].set_visible(False)

    ax.set_ylim(
        0,
        max_stack * 1.25
    )


    # --------------------------------------------------------
    # LEGEND
    # --------------------------------------------------------

    ax.legend(
        title="กลุ่มผู้ใช้ประโยชน์",
        bbox_to_anchor=(
            1.01,
            1
        ),
        loc="upper left",
        frameon=False,
        fontsize=9,
        title_fontsize=10
    )


    # --------------------------------------------------------
    # NOTE
    # --------------------------------------------------------

    if note:

        fig.text(
            0.01,
            0.01,
            note,
            fontsize=9,
            color="#555555"
        )

        fig.tight_layout(
            rect=[
                0,
                0.05,
                0.80,
                1
            ]
        )

    else:

        fig.tight_layout(
            rect=[
                0,
                0,
                0.80,
                1
            ]
        )


    fig.savefig(
        OUTPUT_DIR / filename,
        dpi=CHART_STYLE.get(
            "save_dpi",
            240
        ),
        bbox_inches="tight"
    )

    plt.show()


# ============================================================
# PART A
# กลุ่มผู้ใช้ประโยชน์
# จำแนกตามประเภทของการวิจัย
# ============================================================

users_by_research = pd.crosstab(
    user_long[
        research_type_col
    ],
    user_long[
        "กลุ่มผู้ใช้ประโยชน์"
    ]
)


# จำนวนโครงการจริงในแต่ละ research type
research_unique = (
    user_long
    .groupby(
        research_type_col
    )[project_id_col]
    .nunique()
)


# เรียงตามจำนวนโครงการ
research_order = (
    research_unique
    .sort_values(
        ascending=False
    )
    .index
)


users_by_research = (
    users_by_research
    .reindex(
        research_order
    )
)

research_unique = (
    research_unique
    .reindex(
        research_order
    )
)


display(
    users_by_research
)


users_by_research.to_csv(
    OUTPUT_DIR /
    "4_6_1_1_users_by_research_type.csv",
    encoding="utf-8-sig"
)


plot_user_stacked(
    count_table=users_by_research,

    unique_total=research_unique,

    title=(
        "กลุ่มผู้ใช้ประโยชน์\n"
        "จำแนกตามประเภทของการวิจัย"
    ),

    filename=(
        "4_6_1_1_users_by_research_type.png"
    ),

    x_label="ประเภทของการวิจัย",

    figsize=(13, 8),

    note=(
        "หมายเหตุ: กลุ่มผู้ใช้ประโยชน์เป็น Multiple response; "
        "1 โครงการอาจมีผู้ใช้ประโยชน์มากกว่า 1 กลุ่ม "
        "ดังนั้นความสูงรวมของแท่งเป็นจำนวนการจัดกลุ่ม "
        "ส่วนตัวเลขบนยอดแท่งคือจำนวนโครงการจริง"
    )
)


# ============================================================
# PART B
# กลุ่มผู้ใช้ประโยชน์
# จำแนกตามประเภทการวิจัยตามกรอบ บพท.
# ============================================================

pmu_long_users = (
    df[
        [project_id_col]
        + pmu_cols
    ]
    .melt(
        id_vars=[
            project_id_col
        ],
        value_vars=pmu_cols,
        value_name="ประเภทตามกรอบ_บพท."
    )
    .dropna(
        subset=[
            "ประเภทตามกรอบ_บพท."
        ]
    )
)


pmu_long_users[
    "ประเภทตามกรอบ_บพท."
] = (
    pmu_long_users[
        "ประเภทตามกรอบ_บพท."
    ]
    .apply(clean_local_text)
)


pmu_long_users = (
    pmu_long_users
    .drop_duplicates(
        [
            project_id_col,
            "ประเภทตามกรอบ_บพท."
        ]
    )
)


# ------------------------------------------------------------
# Merge user x PMU
# ------------------------------------------------------------

user_pmu = (
    user_long[
        [
            project_id_col,
            "กลุ่มผู้ใช้ประโยชน์"
        ]
    ]
    .merge(
        pmu_long_users[
            [
                project_id_col,
                "ประเภทตามกรอบ_บพท."
            ]
        ],
        on=project_id_col,
        how="inner"
    )
)


# ------------------------------------------------------------
# Crosstab
# ------------------------------------------------------------

users_by_pmu = pd.crosstab(
    user_pmu[
        "ประเภทตามกรอบ_บพท."
    ],
    user_pmu[
        "กลุ่มผู้ใช้ประโยชน์"
    ]
)


pmu_unique = (
    user_pmu
    .groupby(
        "ประเภทตามกรอบ_บพท."
    )[project_id_col]
    .nunique()
)


pmu_order = (
    pmu_unique
    .sort_values(
        ascending=False
    )
    .index
)


users_by_pmu = (
    users_by_pmu
    .reindex(
        pmu_order
    )
)

pmu_unique = (
    pmu_unique
    .reindex(
        pmu_order
    )
)


display(
    users_by_pmu
)


users_by_pmu.to_csv(
    OUTPUT_DIR /
    "4_6_1_1_users_by_pmu_framework.csv",
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# PMU label
# ------------------------------------------------------------

def pmu_display_local(x):

    if "PMU_LABELS" in globals():

        if x in PMU_LABELS:
            return PMU_LABELS[x]

    return wrap_local(
        x,
        width=22
    )


# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

plot_user_stacked(
    count_table=users_by_pmu,

    unique_total=pmu_unique,

    title=(
        "กลุ่มผู้ใช้ประโยชน์\n"
        "จำแนกตามประเภทการวิจัยตามกรอบ บพท."
    ),

    filename=(
        "4_6_1_1_users_by_pmu_framework.png"
    ),

    x_label="ประเภทการวิจัยตามกรอบ บพท.",

    x_label_func=pmu_display_local,

    rotate=25,

    figsize=(16, 9),

    note=(
        "หมายเหตุ: ทั้งกลุ่มผู้ใช้ประโยชน์และประเภทการวิจัยตามกรอบ บพท. "
        "เป็น Multiple response; 1 โครงการอาจอยู่ได้มากกว่า 1 กลุ่ม "
        "ดังนั้นความสูงรวมของแท่งเป็นจำนวนการจัดกลุ่ม "
        "ส่วนตัวเลขบนยอดแท่งคือจำนวนโครงการจริง"
    )
)


print("สร้างไฟล์เรียบร้อย:")
print("- 4_6_1_1_users_by_research_type.png")
print("- 4_6_1_1_users_by_pmu_framework.png")

In [ ]:
# ============================================================
# 4.6.1.3 Crosstab:
# ความสัมพันธ์ระหว่างผลลัพธ์ ผู้ใช้ ผลผลิต
# และประเภทการวิจัยตามกรอบ บพท.
#
# ตารางที่ 1:
# Outcome x User group
#
# ตารางที่ 2:
# Outcome x Output type
#
# ตารางที่ 3:
# Outcome x User group x PMU research framework
#
# ทุก cell = จำนวนโครงการ (unique project)
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# 0. SETTINGS
# ============================================================

project_id_col = "รหัสโครงการ"

pmu_cols = [
    "ประเภทของการวิจัยตามกรอบ บพท. (1) (Gift)",
    "ประเภทของการวิจัยตามกรอบ (2) บพท. (Gift)",
    "ประเภทของการวิจัยตามกรอบ (3) (Gift)"
]


# ============================================================
# 1. ตรวจว่าตัวแปรจากส่วนก่อนหน้ามีอยู่
# ============================================================

required_vars = [
    "outcome_long",
    "user_long",
    "output_long"
]

missing_vars = [
    v for v in required_vars
    if v not in globals()
]

if missing_vars:
    raise ValueError(
        "กรุณารันส่วนก่อนหน้าก่อน พบว่าขาดตัวแปร: "
        + ", ".join(missing_vars)
    )


# ============================================================
# 2. เตรียม Outcome
#
# outcome_long จาก 4.6.1.2 ถูก normalize แล้ว
# ============================================================

outcome_cross = (
    outcome_long[
        [
            project_id_col,
            "ประเภทผลลัพธ์"
        ]
    ]
    .dropna()
    .drop_duplicates()
    .copy()
)


# ============================================================
# 3. เตรียม User
# ============================================================

user_cross = (
    user_long[
        [
            project_id_col,
            "กลุ่มผู้ใช้ประโยชน์"
        ]
    ]
    .dropna()
    .drop_duplicates()
    .copy()
)


# ============================================================
# 4. เตรียม Output
# ============================================================

output_cross = (
    output_long[
        [
            project_id_col,
            "ประเภทผลผลิต"
        ]
    ]
    .dropna()
    .drop_duplicates()
    .copy()
)


# ============================================================
# 5. เตรียม PMU Framework
# ============================================================

pmu_cross = (
    df[
        [project_id_col]
        + pmu_cols
    ]
    .melt(
        id_vars=[
            project_id_col
        ],
        value_vars=pmu_cols,
        value_name="ประเภทการวิจัยตามกรอบ บพท."
    )
    .dropna(
        subset=[
            "ประเภทการวิจัยตามกรอบ บพท."
        ]
    )
)


pmu_cross[
    "ประเภทการวิจัยตามกรอบ บพท."
] = (
    pmu_cross[
        "ประเภทการวิจัยตามกรอบ บพท."
    ]
    .astype(str)
    .str.strip()
)


pmu_cross = (
    pmu_cross
    .drop_duplicates(
        subset=[
            project_id_col,
            "ประเภทการวิจัยตามกรอบ บพท."
        ]
    )
)


# ============================================================
# 6. TABLE 1
#
# ผลลัพธ์ × กลุ่มผู้ใช้ประโยชน์
# ============================================================

outcome_user_long = (
    outcome_cross
    .merge(
        user_cross,
        on=project_id_col,
        how="inner"
    )
    .drop_duplicates(
        subset=[
            project_id_col,
            "ประเภทผลลัพธ์",
            "กลุ่มผู้ใช้ประโยชน์"
        ]
    )
)


# ------------------------------------------------------------
# Crosstab
# แต่ละ cell = จำนวน unique project
# ------------------------------------------------------------

ct_outcome_user = pd.crosstab(
    index=outcome_user_long[
        "ประเภทผลลัพธ์"
    ],

    columns=outcome_user_long[
        "กลุ่มผู้ใช้ประโยชน์"
    ],

    values=outcome_user_long[
        project_id_col
    ],

    aggfunc=pd.Series.nunique
).fillna(0).astype(int)


# เรียง Outcome จากจำนวนรวมมาก -> น้อย
ct_outcome_user[
    "รวม"
] = (
    ct_outcome_user
    .sum(axis=1)
)

ct_outcome_user = (
    ct_outcome_user
    .sort_values(
        "รวม",
        ascending=False
    )
)


display(
    ct_outcome_user
)


ct_outcome_user.to_csv(
    OUTPUT_DIR /
    "4_6_1_3_outcome_by_user_crosstab.csv",
    encoding="utf-8-sig"
)


# ============================================================
# 7. TABLE 2
#
# ผลลัพธ์ × ประเภทผลผลิต
# ============================================================

outcome_output_long = (
    outcome_cross
    .merge(
        output_cross,
        on=project_id_col,
        how="inner"
    )
    .drop_duplicates(
        subset=[
            project_id_col,
            "ประเภทผลลัพธ์",
            "ประเภทผลผลิต"
        ]
    )
)


ct_outcome_output = pd.crosstab(
    index=outcome_output_long[
        "ประเภทผลลัพธ์"
    ],

    columns=outcome_output_long[
        "ประเภทผลผลิต"
    ],

    values=outcome_output_long[
        project_id_col
    ],

    aggfunc=pd.Series.nunique
).fillna(0).astype(int)


# รวมจำนวน association
ct_outcome_output[
    "รวม"
] = (
    ct_outcome_output
    .sum(axis=1)
)


ct_outcome_output = (
    ct_outcome_output
    .sort_values(
        "รวม",
        ascending=False
    )
)


display(
    ct_outcome_output
)


ct_outcome_output.to_csv(
    OUTPUT_DIR /
    "4_6_1_3_outcome_by_output_crosstab.csv",
    encoding="utf-8-sig"
)


# ============================================================
# 8. TABLE 3
#
# ผลลัพธ์ × กลุ่มผู้ใช้ประโยชน์
# จำแนกตามประเภทการวิจัยตามกรอบ บพท.
#
# Columns จะเป็น MultiIndex:
# PMU framework
#     └── User group
# ============================================================

outcome_user_pmu_long = (
    outcome_cross

    .merge(
        user_cross,
        on=project_id_col,
        how="inner"
    )

    .merge(
        pmu_cross,
        on=project_id_col,
        how="inner"
    )

    .drop_duplicates(
        subset=[
            project_id_col,
            "ประเภทผลลัพธ์",
            "กลุ่มผู้ใช้ประโยชน์",
            "ประเภทการวิจัยตามกรอบ บพท."
        ]
    )
)


ct_outcome_user_pmu = pd.crosstab(
    index=outcome_user_pmu_long[
        "ประเภทผลลัพธ์"
    ],

    columns=[
        outcome_user_pmu_long[
            "ประเภทการวิจัยตามกรอบ บพท."
        ],

        outcome_user_pmu_long[
            "กลุ่มผู้ใช้ประโยชน์"
        ]
    ],

    values=outcome_user_pmu_long[
        project_id_col
    ],

    aggfunc=pd.Series.nunique
).fillna(0).astype(int)


# ------------------------------------------------------------
# เรียง row โดยใช้ผลรวม
# แต่ไม่เพิ่ม "รวม" เข้า MultiIndex
# ------------------------------------------------------------

row_total = (
    ct_outcome_user_pmu
    .sum(axis=1)
)


ct_outcome_user_pmu = (
    ct_outcome_user_pmu
    .loc[
        row_total
        .sort_values(
            ascending=False
        )
        .index
    ]
)


display(
    ct_outcome_user_pmu
)


ct_outcome_user_pmu.to_csv(
    OUTPUT_DIR /
    "4_6_1_3_outcome_by_user_pmu_crosstab.csv",
    encoding="utf-8-sig"
)


# ============================================================
# 9. TABLE 4 — เพิ่มให้สำหรับการวิเคราะห์
#
# Outcome × PMU framework
#
# ตารางนี้อ่านง่ายกว่าตาราง 3 มาก
# และเหมาะสำหรับเอาเข้าเล่มรายงาน
# ============================================================

outcome_pmu_long = (
    outcome_cross
    .merge(
        pmu_cross,
        on=project_id_col,
        how="inner"
    )
    .drop_duplicates(
        subset=[
            project_id_col,
            "ประเภทผลลัพธ์",
            "ประเภทการวิจัยตามกรอบ บพท."
        ]
    )
)


ct_outcome_pmu = pd.crosstab(
    index=outcome_pmu_long[
        "ประเภทผลลัพธ์"
    ],

    columns=outcome_pmu_long[
        "ประเภทการวิจัยตามกรอบ บพท."
    ],

    values=outcome_pmu_long[
        project_id_col
    ],

    aggfunc=pd.Series.nunique
).fillna(0).astype(int)


ct_outcome_pmu[
    "รวม"
] = (
    ct_outcome_pmu
    .sum(axis=1)
)


ct_outcome_pmu = (
    ct_outcome_pmu
    .sort_values(
        "รวม",
        ascending=False
    )
)


display(
    ct_outcome_pmu
)


ct_outcome_pmu.to_csv(
    OUTPUT_DIR /
    "4_6_1_3_outcome_by_pmu_crosstab.csv",
    encoding="utf-8-sig"
)


# ============================================================
# 10. SUMMARY
# ============================================================

print(
    "สร้าง Crosstab เรียบร้อย"
)

print(
    "1. Outcome × User group:",
    ct_outcome_user.shape
)

print(
    "2. Outcome × Output type:",
    ct_outcome_output.shape
)

print(
    "3. Outcome × User group × PMU framework:",
    ct_outcome_user_pmu.shape
)

print(
    "4. Outcome × PMU framework:",
    ct_outcome_pmu.shape
)

In [ ]:
# ============================================================
# 4.6.1.3 — REPORT-READY VISUALIZATION
#
# 4_6_1_3_1 Outcome × User group      -> Heatmap
# 4_6_1_3_2 Outcome × Output type     -> Heatmap
# 4_6_1_3_3 Outcome × PMU framework   -> Grouped horizontal bar
#
# ใช้ Crosstab ที่สร้างไว้ก่อนหน้า:
# ct_outcome_user
# ct_outcome_output
# ct_outcome_pmu
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import textwrap
from matplotlib.colors import LinearSegmentedColormap


# ============================================================
# 0. ตรวจตัวแปร
# ============================================================

required_tables = [
    "ct_outcome_user",
    "ct_outcome_output",
    "ct_outcome_pmu"
]

missing_tables = [
    x for x in required_tables
    if x not in globals()
]

if missing_tables:
    raise ValueError(
        "กรุณารัน Crosstab 4.6.1.3 ก่อน ขาดตัวแปร: "
        + ", ".join(missing_tables)
    )


# ============================================================
# 1. HELPER
# ============================================================

def wrap_report_label(text, width=22):

    if pd.isna(text):
        return ""

    return "\n".join(
        textwrap.wrap(
            str(text),
            width=width,
            break_long_words=False,
            break_on_hyphens=False
        )
    )


# ------------------------------------------------------------
# Outcome display labels
# ------------------------------------------------------------

OUTCOME_REPORT_LABELS = {

    "เพิ่มผลิตภาพการผลิต/เพิ่มผลผลิต/เพิ่มประสิทธิภาพการดำเนินงาน":
        "เพิ่มผลิตภาพ/เพิ่มผลผลิต/\nเพิ่มประสิทธิภาพ",

    "สร้างเครือข่ายและความร่วมมือทางสังคม":
        "สร้างเครือข่ายและ\nความร่วมมือทางสังคม",

    "ลดความสูญเสียทางเศรษฐกิจและสังคม":
        "ลดความสูญเสียทาง\nเศรษฐกิจและสังคม",

    "พัฒนาคุณภาพชีวิต":
        "พัฒนาคุณภาพชีวิต",

    "พัฒนาชุมชน":
        "พัฒนาชุมชน",

    "เพิ่มรายได้":
        "เพิ่มรายได้",

    "ลดต้นทุนการผลิต":
        "ลดต้นทุนการผลิต",

    "ลดขยะและของเสีย":
        "ลดขยะและของเสีย",

    "คุณภาพสิ่งแวดล้อมดีขึ้น ลดมลพิษ/มลภาวะ/ลดก๊าซเรือนกระจก":
        "คุณภาพสิ่งแวดล้อมดีขึ้น/\nลดมลพิษและก๊าซเรือนกระจก",
}


def outcome_report_label(x):

    x = str(x)

    if x in OUTCOME_REPORT_LABELS:
        return OUTCOME_REPORT_LABELS[x]

    return wrap_report_label(
        x,
        width=24
    )


# ------------------------------------------------------------
# User display labels
# ------------------------------------------------------------

USER_REPORT_LABELS = {

    "หน่วยงานภาครัฐในระดับพื้นที่ ตำบล อำเภอ จังหวัด":
        "หน่วยงานรัฐ\nระดับพื้นที่",

    "องค์กรปกครองส่วนท้องถิ่น":
        "องค์กรปกครอง\nส่วนท้องถิ่น",

    "ชุมชน/เครือข่าย":
        "ชุมชน/\nเครือข่าย",

    "ประชาชนกลุ่มเป้าหมาย":
        "ประชาชน\nกลุ่มเป้าหมาย",

    "โรงพยาบาล/สถานพยาบาล":
        "โรงพยาบาล/\nสถานพยาบาล",

    "ผู้ป่วย":
        "ผู้ป่วย",

    "หน่วยงานภาครัฐระดับส่วนกลาง":
        "หน่วยงานรัฐ\nส่วนกลาง",

    "บุคลากรทางการศึกษา/สถาบันการศึกษา":
        "บุคลากร/สถาบัน\nการศึกษา",
}


def user_report_label(x):

    x = str(x)

    if x in USER_REPORT_LABELS:
        return USER_REPORT_LABELS[x]

    return wrap_report_label(
        x,
        width=18
    )


# ------------------------------------------------------------
# Output display labels
# ------------------------------------------------------------

OUTPUT_REPORT_LABELS = {

    "ฐานข้อมูล ระบบและกลไก":
        "ฐานข้อมูล\nระบบและกลไก",

    "กำลังคน หรือหน่วยงาน ที่ได้รับการพัฒนาทักษะ":
        "กำลังคน/หน่วยงาน\nที่พัฒนาทักษะ",

    "นวัตกรรมทางสังคม":
        "นวัตกรรม\nทางสังคม",

    "เครือข่าย":
        "เครือข่าย",

    "อื่นๆ":
        "อื่นๆ",

    "ข้อเสนอแนะเชิงนโยบาย":
        "ข้อเสนอแนะ\nเชิงนโยบาย",

    "ข้อเสนอแนะเชิงนโยบาย:ข้อเสนอที่มุ่งใช้ประกอบการตัดสินใจ การกำหนดนโยบาย มาตรการ แผนงาน แนวทางปฏิบัติ หรือกฎเกณฑ์ของหน่วยงานหรือองค์กร":
        "ข้อเสนอแนะ\nเชิงนโยบาย*",

    "ต้นแบบผลิตภัณฑ์":
        "ต้นแบบ\nผลิตภัณฑ์",

    "เครื่องมือ และโครงสร้างพื้นฐานที่สร้างขึ้น หรือพัฒนาต่อยอดภายใต้โครงการ":
        "เครื่องมือ/\nโครงสร้างพื้นฐาน",

    "เทคโนโลยี/กระบวนการใหม่":
        "เทคโนโลยี/\nกระบวนการใหม่",
}


def output_report_label(x):

    x = str(x)

    if x in OUTPUT_REPORT_LABELS:
        return OUTPUT_REPORT_LABELS[x]

    return wrap_report_label(
        x,
        width=18
    )


# ------------------------------------------------------------
# PMU display label
# ------------------------------------------------------------

def pmu_report_label(x):

    if "PMU_LABELS" in globals():

        if x in PMU_LABELS:
            return wrap_report_label(
                PMU_LABELS[x],
                width=24
            )

    return wrap_report_label(
        x,
        width=24
    )


# ============================================================
# 2. Pastel heatmap color map
# ============================================================

pastel_heatmap = LinearSegmentedColormap.from_list(
    "pastel_heatmap",
    [
        "#FFFFFF",
        "#EAF3FA",
        "#C9DFF0",
        "#A8D5BA",
        "#F6C28B"
    ]
)


# ============================================================
# 3. HELPER: HEATMAP
# ============================================================

def plot_report_heatmap(
    table,
    title,
    filename,
    row_label_func,
    col_label_func,
    note,
    figsize=(15, 9)
):

    plot_df = table.copy()

    # เอาคอลัมน์ "รวม" ออก
    if "รวม" in plot_df.columns:
        plot_df = plot_df.drop(
            columns=["รวม"]
        )

    # เรียง row ตามผลรวม
    row_total = (
        plot_df
        .sum(axis=1)
    )

    plot_df = (
        plot_df
        .loc[
            row_total
            .sort_values(
                ascending=False
            )
            .index
        ]
    )

    # เรียง column ตามผลรวม
    col_total = (
        plot_df
        .sum(axis=0)
    )

    plot_df = (
        plot_df[
            col_total
            .sort_values(
                ascending=False
            )
            .index
        ]
    )

    data = plot_df.values.astype(float)

    fig, ax = plt.subplots(
        figsize=figsize
    )

    im = ax.imshow(
        data,
        cmap=pastel_heatmap,
        aspect="auto",
        interpolation="nearest"
    )

    # --------------------------------------------------------
    # Tick labels
    # --------------------------------------------------------

    row_labels = [
        row_label_func(x)
        for x in plot_df.index
    ]

    col_labels = [
        col_label_func(x)
        for x in plot_df.columns
    ]

    ax.set_yticks(
        np.arange(
            len(row_labels)
        )
    )

    ax.set_yticklabels(
        row_labels,
        fontsize=10
    )

    ax.set_xticks(
        np.arange(
            len(col_labels)
        )
    )

    ax.set_xticklabels(
        col_labels,
        fontsize=9
    )

    plt.setp(
        ax.get_xticklabels(),
        rotation=32,
        ha="right",
        rotation_mode="anchor"
    )

    # --------------------------------------------------------
    # ใส่จำนวนใน cell
    # --------------------------------------------------------

    max_value = (
        np.nanmax(data)
        if data.size > 0
        else 1
    )

    for i in range(
        data.shape[0]
    ):

        for j in range(
            data.shape[1]
        ):

            value = int(
                data[i, j]
            )

            if value == 0:
                label = ""
            else:
                label = str(value)

            # ถ้าสีเข้มขึ้น เปลี่ยน text เป็นขาว
            if max_value > 0 and data[i, j] >= max_value * 0.65:
                text_color = "white"
            else:
                text_color = "#333333"

            ax.text(
                j,
                i,
                label,
                ha="center",
                va="center",
                fontsize=9,
                fontweight=(
                    "bold"
                    if value > 0
                    else "normal"
                ),
                color=text_color
            )

    # --------------------------------------------------------
    # เส้นแบ่ง cell
    # --------------------------------------------------------

    ax.set_xticks(
        np.arange(
            -0.5,
            len(col_labels),
            1
        ),
        minor=True
    )

    ax.set_yticks(
        np.arange(
            -0.5,
            len(row_labels),
            1
        ),
        minor=True
    )

    ax.grid(
        which="minor",
        color="white",
        linewidth=1.5
    )

    ax.tick_params(
        which="minor",
        bottom=False,
        left=False
    )

    # --------------------------------------------------------
    # Title
    # --------------------------------------------------------

    ax.set_title(
        title,
        fontsize=18,
        fontweight="bold",
        pad=18
    )

    ax.set_xlabel("")
    ax.set_ylabel("")

    ax.spines[
        ["top", "right", "bottom", "left"]
    ].set_visible(False)

    # --------------------------------------------------------
    # Colorbar
    # --------------------------------------------------------

    cbar = fig.colorbar(
        im,
        ax=ax,
        fraction=0.025,
        pad=0.02
    )

    cbar.set_label(
        "จำนวนโครงการ",
        fontsize=10
    )

    # --------------------------------------------------------
    # Note
    # --------------------------------------------------------

    fig.text(
        0.01,
        0.01,
        note,
        fontsize=9,
        color="#555555"
    )

    fig.tight_layout(
        rect=[
            0,
            0.05,
            1,
            1
        ]
    )

    fig.savefig(
        OUTPUT_DIR / filename,
        dpi=CHART_STYLE.get(
            "save_dpi",
            240
        ),
        bbox_inches="tight"
    )

    plt.show()


# ============================================================
# 4.6.1.3-1
# Outcome × User group
# ============================================================

plot_report_heatmap(

    table=ct_outcome_user,

    title=(
        "ความสัมพันธ์ระหว่างผลลัพธ์ของโครงการ\n"
        "กับกลุ่มผู้ใช้ประโยชน์"
    ),

    filename=(
        "4_6_1_3_1_outcome_user_heatmap.png"
    ),

    row_label_func=outcome_report_label,

    col_label_func=user_report_label,

    note=(
        "หมายเหตุ: ตัวเลขในแต่ละช่องแสดงจำนวนโครงการที่พบผลลัพธ์และ"
        "กลุ่มผู้ใช้ประโยชน์นั้นร่วมกัน; ทั้งผลลัพธ์และผู้ใช้ประโยชน์"
        "เป็น Multiple response"
    ),

    figsize=(
        max(
            15,
            len(ct_outcome_user.columns) * 1.5
        ),
        max(
            8,
            len(ct_outcome_user) * 0.75 + 2
        )
    )
)


# ============================================================
# 4.6.1.3-2
# Outcome × Output type
# ============================================================

plot_report_heatmap(

    table=ct_outcome_output,

    title=(
        "ความสัมพันธ์ระหว่างผลลัพธ์ของโครงการ\n"
        "กับประเภทผลผลิต"
    ),

    filename=(
        "4_6_1_3_2_outcome_output_heatmap.png"
    ),

    row_label_func=outcome_report_label,

    col_label_func=output_report_label,

    note=(
        "หมายเหตุ: ตัวเลขในแต่ละช่องแสดงจำนวนโครงการที่พบผลลัพธ์และ"
        "ผลผลิตประเภทนั้นร่วมกัน; ทั้งผลลัพธ์และประเภทผลผลิต"
        "เป็น Multiple response"
    ),

    figsize=(
        max(
            16,
            len(ct_outcome_output.columns) * 1.45
        ),
        max(
            8,
            len(ct_outcome_output) * 0.75 + 2
        )
    )
)


# ============================================================
# 4.6.1.3-3
# Outcome × PMU framework
# Horizontal grouped bar — report-ready compact version
# ============================================================

import numpy as np
import matplotlib.pyplot as plt


# ============================================================
# 1. เตรียมข้อมูล
# ============================================================

pmu_plot = ct_outcome_pmu.copy()

# เอาคอลัมน์ "รวม" ออก
if "รวม" in pmu_plot.columns:
    pmu_plot = pmu_plot.drop(columns=["รวม"])


# ------------------------------------------------------------
# เรียง Outcome ตามจำนวนรวม
# มากอยู่ด้านบน
# ------------------------------------------------------------

row_total = pmu_plot.sum(axis=1)

pmu_plot = pmu_plot.loc[
    row_total
    .sort_values(ascending=True)
    .index
]


# ------------------------------------------------------------
# เรียง PMU ตามจำนวนรวม
# ------------------------------------------------------------

col_order = (
    pmu_plot
    .sum(axis=0)
    .sort_values(ascending=False)
    .index
)

pmu_plot = pmu_plot[col_order]


# ============================================================
# 2. สี pastel
# ============================================================

pmu_palette_report = [
    "#8FB9E0",  # ฟ้า
    "#A8D5BA",  # เขียว
    "#F6C28B",  # พีช
    "#C7B6E5",  # ม่วง
    "#F2B5B5",  # ชมพู
    "#9ED9CC",  # มิ้นต์
    "#F7D6A3",  # ครีม
    "#AFCBFF",
]

colors = [
    pmu_palette_report[i % len(pmu_palette_report)]
    for i in range(len(pmu_plot.columns))
]


# ============================================================
# 3. Figure
# ============================================================

fig_height = max(
    7.5,
    len(pmu_plot) * 0.82 + 1.8
)

fig, ax = plt.subplots(
    figsize=(12.5, fig_height)
)


# ============================================================
# 4. พิกัด
# ============================================================

y = np.arange(len(pmu_plot))

n_series = len(pmu_plot.columns)

group_height = 0.82

bar_height = (
    group_height
    / max(n_series, 1)
)


# ============================================================
# 5. วาด grouped horizontal bars
# ============================================================

for j, framework in enumerate(pmu_plot.columns):

    offset = (
        j
        - (n_series - 1) / 2
    ) * bar_height

    values = pmu_plot[framework].values

    bars = ax.barh(
        y + offset,
        values,
        height=bar_height * 0.92,
        color=colors[j],
        edgecolor="white",
        linewidth=0.8,
        label=pmu_report_label(framework)
    )

    # --------------------------------------------------------
    # ตัวเลขปลายแท่ง
    # --------------------------------------------------------

    for bar, value in zip(bars, values):

        if value <= 0:
            continue

        ax.text(
            bar.get_width() + 0.12,
            bar.get_y() + bar.get_height()/2,
            str(int(value)),
            ha="left",
            va="center",
            fontsize=9,
            color="#333333"
        )


# ============================================================
# 6. Y labels
# ============================================================

ax.set_yticks(y)

ax.set_yticklabels(
    [
        outcome_report_label(x)
        for x in pmu_plot.index
    ],
    fontsize=10.5
)


# ============================================================
# 7. Title / Axis
# ============================================================

ax.set_title(
    "ผลลัพธ์ของโครงการ\n"
    "จำแนกตามประเภทการวิจัยตามกรอบ บพท.",
    fontsize=18,
    fontweight="bold",
    pad=16
)

ax.set_xlabel(
    "จำนวนโครงการ",
    fontsize=12
)

ax.set_ylabel("")


# ============================================================
# 8. Grid / Style
# ============================================================

ax.grid(
    axis="x",
    linestyle="--",
    alpha=0.18
)

ax.set_axisbelow(True)

ax.spines[
    ["top", "right"]
].set_visible(False)


# ============================================================
# 9. X limit
# ลดพื้นที่ว่างด้านขวา
# ============================================================

maxv = max(
    pmu_plot
    .to_numpy()
    .max(),
    1
)

ax.set_xlim(
    0,
    maxv * 1.13 + 0.7
)


# ============================================================
# 10. Legend ด้านล่าง
# ============================================================

ax.legend(
    title="ประเภทการวิจัยตามกรอบ บพท.",
    loc="upper center",
    bbox_to_anchor=(0.5, -0.12),
    ncol=min(
        2,
        len(pmu_plot.columns)
    ),
    frameon=False,
    fontsize=9,
    title_fontsize=10
)


# ============================================================
# 11. Note
# ============================================================

fig.text(
    0.01,
    0.01,
    (
        "หมายเหตุ: ตัวเลขแสดงจำนวนโครงการที่พบผลลัพธ์และประเภทการวิจัย"
        "ตามกรอบ บพท. นั้นร่วมกัน; ทั้งผลลัพธ์และประเภทการวิจัยตามกรอบ "
        "บพท. เป็น Multiple response"
    ),
    fontsize=9,
    color="#555555"
)


# ============================================================
# 12. Layout
# ============================================================

fig.tight_layout(
    rect=[
        0,
        0.11,
        1,
        1
    ]
)


# ============================================================
# 13. Save
# ============================================================

fig.savefig(
    OUTPUT_DIR /
    "4_6_1_3_3_outcome_pmu_grouped_bar.png",
    dpi=CHART_STYLE.get(
        "save_dpi",
        240
    ),
    bbox_inches="tight"
)

plt.show()


# ============================================================
# FINISH
# ============================================================

print(
    "สร้างภาพสำหรับรายงานเรียบร้อย:"
)

print(
    "- 4_6_1_3_1_outcome_user_heatmap.png"
)

print(
    "- 4_6_1_3_2_outcome_output_heatmap.png"
)

print(
    "- 4_6_1_3_3_outcome_pmu_grouped_bar.png"
)

# 4.7 การประเมินผลสำเร็จตามเกณฑ์ OECD

วิเคราะห์ 6 มิติ:
1. Relevance
2. Coherence
3. Effectiveness
4. Efficiency
5. Outcomes & Impact
6. Sustainability

สำหรับกราฟใยแมงมุมด้านล่าง:
- 4 มิติแรกใช้ **สัดส่วนโครงการที่ได้ค่า 1**
- Outcomes & Impact ใช้ค่า 0–3 แล้วหาร 3 เพื่อ normalize เป็น 0–1 และ **ตัดรหัส 4 = N/A ออก**
- Sustainability ใช้ระดับ 1–3 โดย **1 = ยั่งยืนกว่า, 3 = ไม่ยั่งยืนกว่า** จึงแปลงเป็น `(3-score)/2`; รหัส 4 = N/A ถูกตัดออก
- กราฟนี้เป็น **descriptive normalized summary** เพื่อสื่อสารภาพรวม ไม่ใช่คะแนน OECD มาตรฐานอย่างเป็นทางการ


In [ ]:
binary_oecd = {
    "Relevance": "Relevance (0-ไม่มี, 1 -มี)",
    "Coherence": "Coherence (0-ไม่มี, 1 -มี)",
    "Effectiveness": "Effectiveness (0-ไม่มี, 1 -มี)",
    "Efficiency": "Efficiency (0-ไม่มี, 1 -มี)",
}

oecd_summary_rows = []
for label, col in binary_oecd.items():
    vals = pd.to_numeric(df[col], errors="coerce")
    n1 = vals.eq(1).sum()
    valid = vals.notna().sum()
    oecd_summary_rows.append([label, n1, valid, n1/valid*100 if valid else np.nan])

oecd_binary_table = pd.DataFrame(
    oecd_summary_rows,
    columns=["เกณฑ์", "จำนวนผ่าน", "จำนวนที่มีข้อมูล", "ร้อยละผ่าน"]
).set_index("เกณฑ์")
display(oecd_binary_table.round(2))

# Distribution: Outcome & Impact
oi_score = numeric_prefix(df["Outcomes and Impact (Dropdownlist)"])
oi_table = frequency_table(oi_score, denom=N, dropna=False, sort=False)
display(Markdown("### Outcomes & Impact"))
display(oi_table)

# Distribution: Sustainability
sus_score = numeric_prefix(df["Sustainability (Dropdownlist)"])
sus_table = frequency_table(sus_score, denom=N, dropna=False, sort=False)
display(Markdown("### Sustainability"))
display(sus_table)

# Radar normalized summary
radar_labels = ["Relevance", "Coherence", "Effectiveness", "Efficiency", "Outcome & Impact", "Sustainability"]

radar_values = []
for label in radar_labels[:4]:
    radar_values.append(oecd_binary_table.loc[label, "ร้อยละผ่าน"]/100)

oi_valid = oi_score[oi_score.isin([0,1,2,3])]
radar_values.append((oi_valid/3).mean() if len(oi_valid) else np.nan)

sus_valid = sus_score[sus_score.isin([1,2,3])]
radar_values.append(((3-sus_valid)/2).mean() if len(sus_valid) else np.nan)

radar_df = pd.DataFrame({"มิติ": radar_labels, "ค่าปรับมาตรฐาน 0–1": radar_values})
display(radar_df.round(3))

angles = np.linspace(0, 2*np.pi, len(radar_labels), endpoint=False).tolist()
values = radar_values + radar_values[:1]
angles_closed = angles + angles[:1]

fig = plt.figure(figsize=(7.5, 7.5))
ax = fig.add_subplot(111, polar=True)
ax.plot(angles_closed, values, linewidth=2, marker="o")
ax.fill(angles_closed, values, alpha=0.10)
ax.set_xticks(angles)
ax.set_xticklabels(radar_labels)
ax.set_ylim(0, 1)
ax.set_yticks([0.2,0.4,0.6,0.8,1.0])
ax.set_yticklabels(["0.2","0.4","0.6","0.8","1.0"])
ax.set_title("ภาพรวมผลสำเร็จตามเกณฑ์ OECD\n(Descriptive normalized summary)", pad=24, weight="bold")
fig.tight_layout()
fig.savefig(OUTPUT_DIR/"4_7_oecd_radar.png", bbox_inches="tight")
plt.show()


## ตารางสรุปสำหรับนำไปเขียนบทที่ 4

Cell ด้านล่างสร้างตารางสรุปตัวเลขหลักแบบสั้น เพื่อคัดลอกไปใช้เขียน narrative ได้ทันที


In [ ]:
summary_rows = [
    ["จำนวนโครงการทั้งหมด", N, "100.0%"],
    ["งบประมาณรวม (ล้านบาท)", round(budget.sum()/1e6, 2), ""],
    ["งบประมาณเฉลี่ยต่อโครงการ (ล้านบาท)", round(budget.mean()/1e6, 2), ""],
    ["โครงการขยายเวลา", int(extended.sum()), f"{extended.mean()*100:.1f}%"],
    ["โครงการยุติ", int(terminated.sum()), f"{terminated.mean()*100:.1f}%"],
    ["นักวิจัยรวม (คน)", int(researcher_n.sum()), ""],
    ["นักวิจัยเฉลี่ยต่อโครงการ (คน)", round(researcher_n.mean(), 2), ""],
]

chapter4_summary = pd.DataFrame(summary_rows, columns=["ตัวชี้วัด", "จำนวน/ค่า", "ร้อยละ"])
display(chapter4_summary)
chapter4_summary.to_csv(OUTPUT_DIR/"chapter4_key_summary.csv", index=False, encoding="utf-8-sig")

print(f"บันทึกกราฟและตารางไว้ที่: {OUTPUT_DIR.resolve()}")


# หมายเหตุสำหรับการเขียนรายงาน

- ทุกกราฟควรใช้ชื่อเดียวกับหัวข้อในบทที่ 4 และใส่ `n (%)` ให้สอดคล้องกันทั้งเล่ม
- กราฟ Multiple response ต้องมีเชิงอรรถว่า **“โครงการหนึ่งสามารถอยู่ได้มากกว่า 1 ประเภท จึงรวมร้อยละเกิน 100% ได้”**
- Impact ต้องแยกถ้อยคำ **Actual / Expected** ในการเขียน narrative จาก Note ของแต่ละโครงการ ไม่ควรสรุปค่า 1 ทั้งหมดเป็นผลกระทบที่เกิดขึ้นจริง
- ส่วนผลประโยชน์วิชาการระดับ Outcome เรื่อง citation ยังไม่มีตัวแปรตรงในชีตนี้ จึงควรระบุว่า “ไม่มีข้อมูลสำหรับวิเคราะห์” หรือเติมจากแหล่งข้อมูลอื่นภายหลัง
- หากต้องการรวมคำตอบที่สะกดต่างกันแต่มีความหมายเดียวกัน ควรทำ **canonical mapping ที่ตรวจสอบโดยผู้วิจัย** ก่อนสร้างกราฟฉบับเผยแพร่
